# Stage 2a - SmolVLM-500M LoRA fine-tuning and evaluation

Fine-tunes `HuggingFaceTB/SmolVLM-500M-Instruct` with LoRA on the Stage-1 captions, then evaluates on the 3,316-image test split (BERTScore / ROUGE / BLEURT / entailment) and audits BERTScore by Fitzpatrick skin-tone group.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION - edit this cell only.
#
# These notebooks were developed in Google Colab with the dataset on Google
# Drive, so every path below defaults to a mounted-Drive layout
# (/content/drive/MyDrive/...). Nothing else in the notebook hardcodes a path.
# To run elsewhere, either set the DERM_* environment variables or edit the
# fallback strings, and skip the drive.mount() cell.
# ---------------------------------------------------------------------------
import os
from pathlib import Path

DRIVE_ROOT    = os.environ.get("DERM_DRIVE_ROOT", "/content/drive/MyDrive")
WORK_DIR      = os.environ.get("DERM_WORK_DIR", "/content")

DATASETS_ROOT = f"{DRIVE_ROOT}/Skin_Concepts/Skin_Concepts_datasets"
DATA_ROOT     = f"{DATASETS_ROOT}/fitzpatrick17k/data"   # train/val/test CSVs + image folders
BASE          = f"{DATA_ROOT}/finalfitz17k"              # alt. layout used by some cells
CAPTIONS_DIR  = f"{DATASETS_ROOT}/captions_qwen_rag"     # Stage-1 caption CSVs
RUNS_DIR      = f"{DRIVE_ROOT}/SmolVLM_runs"             # checkpoints / logs / eval CSVs
EVAL_DIR      = f"{DRIVE_ROOT}/Fitz"                     # held-out eval predictions + references

TEST_IMG_DIR = f"{CAPTIONS_DIR}/Test-image"
IMG_ROOT     = f"{EVAL_DIR}/Test-image"
print("RUNS_DIR:", RUNS_DIR, "| EVAL_DIR:", EVAL_DIR)


# Setup

We first setup the environment with the primary necessary libraries and login into Hugging Face.

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install flash-attn --no-build-isolation


Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 123.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Requirement alrea
[... 3822 characters of output trimmed for repo size ...]

In [ ]:
!python -m pip install --upgrade pip -q
!pip install -qU transformers
!pip install -qU accelerate datasets peft bitsandbytes hf_transfer tensorboard

# Can be a good idea to re-start the kernel after this

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 81.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.


In [ ]:
!pip install ninja -qU

In [ ]:
!pip install flash_attn==2.7.3 -q

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'flash_attn' when getting requirements to build wheel


In [ ]:
# !pip freeze > requirements.txt

In [ ]:
# Hugging Face auth. Set HF_TOKEN in your environment (or as a Colab secret)
# before running - needed only for gated models and for pushing artefacts.
import os
from huggingface_hub import login

_tok = os.environ.get("HF_TOKEN")
if _tok:
    login(token=_tok)
    print("Logged in to the Hugging Face Hub.")
else:
    print("HF_TOKEN not set - skipping login.")


# Loading the model and the dataset

We load the model from the Hugging Face hub.

Fine-tuning will be done on a small derm dataset.

In [ ]:
# Enable fast weights download and upload
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0" # if you're connected to multiple gpus but only want to use one.

In [ ]:
# !pip uninstall flash-attn -y

In [ ]:
import torch
from PIL import Image
from transformers import Idefics3ForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig

# NOTE THESE MODELS WILL STRUGGLE WITH MORE THAN ONE IMAGE.
# model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
model_id = "HuggingFaceTB/SmolVLM-500M-Instruct"

DEVICE = "cuda" # or "mps" for mac

# Define the quantization configuration with NF4 and double quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,             # Use 4-bit quantization (NF4)
    bnb_4bit_quant_type="nf4",     # Set quantization type to NF4
    bnb_4bit_use_double_quant=True # Enable double quantization
)

model = Idefics3ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, #float16 for colab
    device_map='auto',
    _attn_implementation="flash_attention_2" if DEVICE == "cuda" else "eager",
    # quantization_config=quant_config, # to use quantization
)
processor = AutoProcessor.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
from PIL import Image
from io import BytesIO
import requests
from transformers.image_utils import load_image

# Image URLs
IMG_URLS = [
    "https://picsum.photos/id/237/400/300",
    # "https://picsum.photos/id/231/200/300",
    # "https://picsum.photos/id/27/500/500",
    # "https://picsum.photos/id/17/150/600",
]

images = [load_image(url) for url in IMG_URLS]

# Display each image
for i, img in enumerate(images):
    img.show(title=f"Image {i + 1}")

# Your prompt
PROMPT = "Describe the images, one by one:"

messages = [
    {"role": "user", "content": [
        {"type": "text", "text": PROMPT},
        {"type": "image"},
        # {"type": "image"},
        # {"type": "image"},
        # {"type": "image"},
    ]}
]

input_text = processor.apply_chat_template(messages, add_generation_prompt=True)

print(input_text)

# Prepare the processor inputs
inputs = processor(text=input_text, images=images, return_tensors="pt").to("cuda")

# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts[0])

<|im_start|>User: Describe the images, one by one:<image><end_of_utterance>
Assistant:
User: Describe the images, one by one:




Assistant: A black dog is sitting on a wooden floor and looking up at the camera.


In [ ]:
# processor.tokenizer.model_max_length=4096 # to reduce VRAM
print(processor.tokenizer.model_max_length)

8192


In [ ]:
# # not needed
# processor.tokenizer.pad_token = processor.tokenizer.eos_token

# # not needed
# processor.tokenizer.padding_side = "left"

## Load Dataset

In [ ]:
# # HF example dataset. Note that you need to select the english query - see dataset prep below.
# from datasets import load_dataset

# train_dataset = load_dataset("nielsr/docvqa_1200_examples", split="train")
# train_dataset = train_dataset.remove_columns(['id', 'words', 'bounding_boxes', 'answer'])

# eval_dataset = load_dataset("nielsr/docvqa_1200_examples", split="test")
# eval_dataset = eval_dataset.remove_columns(['id', 'words', 'bounding_boxes', 'answer'])

In [ ]:
from datasets import load_dataset

# load and prepare dataset
ds = load_dataset("racho1/newFitz")

train_dataset = ds["train"]
eval_dataset = ds["test"]

README.md:   0%|          | 0.00/466 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/493M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/514M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11787 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1474 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_dataset
from PIL import Image
import numpy as np

# 1) load
ds = load_dataset("racho1/newFitz")
train_dataset = ds["train"]
eval_dataset  = ds["test"]

# 2) sanitizer: force every image to RGB
def ensure_rgb(example):
    img = example["image"]

    # If it's a file path (rare), open it
    if isinstance(img, str):
        img = Image.open(img)

    # If it's a numpy array, normalize shape to (H, W, 3)
    if isinstance(img, np.ndarray):
        if img.ndim == 2:                # (H, W)
            img = np.stack([img]*3, axis=-1)
        elif img.ndim == 3 and img.shape[-1] == 1:  # (H, W, 1)
            img = np.repeat(img, 3, axis=-1)
        img = Image.fromarray(img.astype(np.uint8))

    # If it's a PIL image, convert modes like L/LA/RGBA/CMYK → RGB
    if isinstance(img, Image.Image) and img.mode != "RGB":
        img = img.convert("RGB")

    example["image"] = img
    return example

# 3) apply (use multiple workers if available)
train_dataset = train_dataset.map(ensure_rgb, num_proc=4)
eval_dataset  = eval_dataset.map(ensure_rgb, num_proc=4)

# 4) quick checks
def count_modes(dset, n=200):
    modes = {}
    for i in range(min(len(dset), n)):
        im = dset[i]["image"]
        m = im.mode if isinstance(im, Image.Image) else "array"
        modes[m] = modes.get(m, 0) + 1
    return modes

print("Train modes (sample):", count_modes(train_dataset))
print("Eval  modes (sample):", count_modes(eval_dataset))
print("Rows:", len(train_dataset), len(eval_dataset))


Map (num_proc=4):   0%|          | 0/11787 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1474 [00:00<?, ? examples/s]

Train modes (sample): {'RGB': 200}
Eval  modes (sample): {'RGB': 200}
Rows: 11787 1474


In [ ]:
train_dataset[0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=461x383>,
 'caption': 'The image shows a chronic inflammatory skin condition affecting the axillary region. The lesion appears to be a chronic inflammatory condition characterized by persistent or recurrent boil-like nodules and abscesses. The nodules are located in the axillary region;  and the surrounding skin shows scarring and a purulent discharge. The lesion is located on the upper arm;  and the surrounding skin shows a significant amount of hair.',
 'split': 'train'}

In [ ]:
# # show one of the images
# train_dataset[10]["image"]

# Evaluation before Training

In [ ]:
import torch
from PIL import Image
from torchvision.transforms.functional import to_pil_image, resize

def run_model_evaluation(model, dataset, num_samples=None, device='cuda', constant_query=None):
    model.eval()
    results = []

    # Limit the dataset if a specific number of samples is provided
    if num_samples is not None:
        dataset = torch.utils.data.Subset(dataset, range(num_samples))

    for example in dataset:
        image = example["image"]
        if constant_query is None:
            query = example["query"]["en"]
        else:
            query = constant_query  # Use the constant query if provided

        # Display a reduced size version of the image
        pil_image = image
        aspect_ratio = pil_image.width / pil_image.height
        new_width = 300
        new_height = int(new_width / aspect_ratio)
        display_image = resize(pil_image, (new_height, new_width))
        display_image.show()  # This will open the image in the default image viewer

        # Construct the message template
        messages = [
            {
                "role": "user",
                "content": [
                    # {"type": "text", "text": "Answer briefly."},
                    {"type": "text", "text": query},
                    {"type": "image"}, # YOU CAN COMMENT THIS OUT IF THERE ARE NO IMAGES.
                    # {"type": "image"}, # ADD A SECOND IMAGE!!! Note that the text must be "image" for every image.
                ]
            }
        ]

        # Apply the chat template to preprocess input
        formatted_prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        print(f"Formatted prompt: {formatted_prompt}")
        text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=[text.strip()], images=[image], return_tensors="pt", padding=True).to(device)
        # inputs = processor(text=[text.strip()], images=[image1,image2], return_tensors="pt", padding=True).to(device) ## ADD A SECOND IMAGE!!!

        # Generate output from the model
        generated_ids = model.generate(**inputs, max_new_tokens=64)
        generated_texts = processor.batch_decode(generated_ids[:, inputs["input_ids"].size(1):], skip_special_tokens=True)

        print(f"Prediction: {generated_texts[0]}\n")

        results.append(generated_texts[0])  # Store the result

    return results

In [ ]:
# Usage
eval_results = run_model_evaluation(model, eval_dataset, num_samples=3, device='cuda', constant_query="What kind of medical images do you see?")
# print(eval_results)

Formatted prompt: <|im_start|>User: What kind of medical images do you see?<image><end_of_utterance>
Assistant:
Prediction:  A person with a red rash on their arm.

Formatted prompt: <|im_start|>User: What kind of medical images do you see?<image><end_of_utterance>
Assistant:
Prediction:  Two different colors of lipstick on someone's arm.

Formatted prompt: <|im_start|>User: What kind of medical images do you see?<image><end_of_utterance>
Assistant:
Prediction:  A close-up of a man's face with a small bump on the forehead.



## Other Sample
The example below shows how you can feed in zero, one or two images to the evaluation.

For no images, you can pass `images=None` into the processor. You can see how this is automatically handled below.

You can also train on multiple images using this approach, by adjusting the datacollator below in the same manner as is done for evaluation here. Note that you'll need to update your dataset so that it has a column for second (and subsequent images) and ensure they are passed correctly within the data collator.

In [ ]:
# # SMOLVLM is poor on multiple images.

# image1 = "http://images.cocodataset.org/val2017/000000039769.jpg"
# image2 = "http://images.cocodataset.org/val2017/000000219578.jpg"

# # Choose between one of the following.
# image_urls = [image1, image2]
# # images = [image1]
# # images = None

# images = [load_image(url) for url in image_urls]

# # Display each image
# for i, img in enumerate(images):
#     img.show(title=f"Image {i + 1}")

# messages = []

# # Add images
# if images is not None:
#     message_content = [{"type": "image"} for _ in images]
#     message_content.append({"type": "text", "text": "How many images do you see and what is in each?"})
# else:
#     message_content = [{"type": "text", "text": "How are you today?"}]

# # Add the composed message
# messages.append({
#     "role": "user",
#     "content": message_content
# })

# print(messages)

# # Or you can do this manually
# # messages = [
# #     {
# #         "role": "user",
# #         "content": [
# #             {"type": "image"}, # ADD a first image
# #             {"type": "image"}, # ADD A SECOND IMAGE!!! Note that the text must be "image" for every image.
# #             {"type": "text", "text": "Say hello"}
# #         ]
# #     }
# # ]

# # Apply the chat template to preprocess input
# text = processor.apply_chat_template(messages, add_generation_prompt=True)

# print(f"formatted prompt: {text}")

# inputs = processor(
#     text=[text.strip()],
#     images=images,
#     return_tensors="pt",
#     padding=True).to('cuda')

# # Generate output from the model
# generated_ids = model.generate(**inputs, max_new_tokens=64, temperature=0.7, do_sample=True)
# generated_texts = processor.batch_decode(generated_ids[:, inputs["input_ids"].size(1):], skip_special_tokens=True)

# print(f"Prediction: {generated_texts[0]}\n")

## Manual Evaluation on an image

### No image splitting

In [ ]:
import torch
from PIL import Image
import requests
from torchvision.transforms.functional import to_pil_image, resize

def evaluate_image(image_url, model, processor, device='cuda'):
    model.eval()

    # Properly handle the image fetch and load
    response = requests.get(image_url, stream=True)  # Ensure the response is streamed
    response.raw.decode_content = True  # Decode the content that was streamed
    pil_image = Image.open(response.raw).convert('RGB')  # Now open it with PIL

    # Resize the image for display
    aspect_ratio = pil_image.width / pil_image.height
    new_width = 600
    new_height = int(new_width / aspect_ratio)
    display_image = resize(pil_image, (new_height, new_width))
    display_image.show()  # Display the image

    # Construct the message template
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"}, # YOU CAN COMMENT THIS OUT IF THERE ARE NO IMAGES.
                {"type": "text", "text": "What headlines do you see here?"}
            ]
        }
    ]

    # Apply the chat template to preprocess input
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text.strip()], images=[pil_image], return_tensors="pt", padding=True).to(device)

    # Generate output from the model
    generated_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.3)
    generated_texts = processor.batch_decode(generated_ids[:, inputs["input_ids"].size(1):], skip_special_tokens=True)

    print(f"Prediction: {generated_texts[0]}")
    return generated_texts[0]

# Example usage
image_url = "https://www.tomorrowspapers.co.uk/wp-content/uploads/2024/01/Financial-Times-18.jpg"
result = evaluate_image(image_url, model, processor)

Prediction:  The headlines include "Farmers' Fury" and "US Economy Grows 3.5% and Inflation Slows as Biden looks for election boost." The text also discusses the impact of the coronavirus (COVID-19) on the economy and the potential for a re-election campaign.


# Training loop

We first define the data collator which takes list of samples and return input tensors fed to the model. There are 4 tensors types we are interested:
- `input_ids`: these are the input indices fed to the language model
- `attention_mask`: the attention mask for the `input_ids` in the language model
- `pixel_values`: the (pre-processed) pixel values that encode the image(s). Idefics2 treats images in their native resolution (up to 980) and their native aspect ratio
- `pixel_attention_mask`: when multiple image(s) are packed into the same sample (or in the batch), attention masks for the images are necessary because of these images can have different sizes and aspect ratio. This masking ensures that the vision encoder properly forwards the images.


In [ ]:
# ONLY COMPUTE THE LOSS OVER ASSISTANT RESPONSES.

import torch

class MyDataCollator:
    def __init__(self, processor):
        self.processor = processor
        # Hard-code a representative substring for the assistant response (a bit hacky but we are taking the last two tokens of the assistant template and masking that and what goes before.
        self.hardcoded_subsequence = processor.tokenizer("\nAssistant:", return_tensors="pt")["input_ids"][0][-2:]

    def __call__(self, examples):
        texts = []
        images = []
        for example in examples:
            image = example["image"]
            question = "What do you see here?"
            answer = example["caption"]

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": question},
                        {"type": "image"},  # Images after the text
                    ],
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": answer},
                    ],
                },
            ]

            # Convert messages to text format using the processor's template
            text = self.processor.apply_chat_template(messages, add_generation_prompt=False)

            texts.append(text.strip())
            images.append([image])

        # Tokenize and process batch
        batch = self.processor(text=texts, images=images, return_tensors="pt", padding=True)

        # Prepare labels: Clone input IDs
        labels = batch["input_ids"].clone()

        # print(f"Hardcoded subsequence is: {self.hardcoded_subsequence}")

        # Mask non-assistant tokens using the hardcoded subsequence
        for i, input_ids in enumerate(batch["input_ids"]):
            start_idx = self.find_hardcoded_subsequence(input_ids)
            if start_idx is not None:
                labels[i, :start_idx + len(self.hardcoded_subsequence)] = -100
                # labels[i, start_idx + len(self.hardcoded_subsequence):] = -100  # Ignore tokens after
            else:
                print(f"Warning: Hardcoded subsequence not found in input {i}.")
                print(labels[i,:])
                labels[i, :] = -100  # Mask the entire sequence if not found

        # Assign masked labels back to the batch
        batch["labels"] = labels
        return batch

    def find_hardcoded_subsequence(self, sequence):
        """Find the start index of the hardcoded subsequence in a tokenized sequence."""
        seq_len = len(sequence)
        sub_len = len(self.hardcoded_subsequence)

        for i in range(seq_len - sub_len + 1):
            if torch.equal(sequence[i:i + sub_len], self.hardcoded_subsequence):
                return i
        return None

data_collator = MyDataCollator(processor)

In [ ]:
#show a first example
print(train_dataset[0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=461x383 at 0x79EFEC235340>, 'caption': 'The image shows a chronic inflammatory skin condition affecting the axillary region. The lesion appears to be a chronic inflammatory condition characterized by persistent or recurrent boil-like nodules and abscesses. The nodules are located in the axillary region;  and the surrounding skin shows scarring and a purulent discharge. The lesion is located on the upper arm;  and the surrounding skin shows a significant amount of hair.', 'split': 'train'}


In [ ]:
# If you want to decode tokens.
# # Tokens to decode
# tokens = [49190, 49189,
#           49279,
#           198, 9519,
#           9531,
#           42,
#           # 330,
#           # 2244, 2537, 651, 2231, 30, 49279
#          ]

# # Decode tokens to text
# decoded_text = processor.tokenizer.decode(tokens, skip_special_tokens=False)

# print(decoded_text)

In [ ]:
# Inspect Dataset

# Select a small batch of examples (e.g., 2 examples for quick testing)
sample_batch = [train_dataset[i] for i in range(2)]

# Call the data collator with the sample batch to process it
processed_batch = data_collator(sample_batch)
assistant_responses = [example["caption"] for example in sample_batch]  # Extract assistant responses manually

# Print the processed batch keys to check what's inside
print("Processed batch keys:", processed_batch.keys())

# Tokenized input IDs
input_ids = processed_batch["input_ids"]
labels = processed_batch["labels"]

# # Print tokenized input IDs
# print("\nTokenized input IDs:")
# print(input_ids)

# # Print labels (masked tokens)
# print("\nLabels (with masking applied):")
# print(labels)

# # Decode the tokenized inputs for human-readable format
# print("\nDecoded input texts:")
# for idx, input_id in enumerate(input_ids):
#     print(f"Example {idx}:")
#     print(processor.tokenizer.decode(input_id, skip_special_tokens=False))

# # Decode the labels to inspect which tokens are contributing to the loss
# print("\nDecoded labels (masking applied):")
# for idx, label in enumerate(labels):
#     # Decode the labels with masking handled
#     decoded_labels = processor.tokenizer.decode(
#         [token for token in label.tolist() if token != -100], skip_special_tokens=False
#     )
#     print(f"Example {idx}:")
#     print("Decoded:", decoded_labels)

# Sanity Check: Assistant Responses vs Decoded Labels
print("\nSanity Check: Assistant Responses vs Decoded Labels")
for idx, assistant_response in enumerate(assistant_responses):
    # Decode the labels again for comparison
    decoded_labels = processor.tokenizer.decode(
        [token for token in labels[idx].tolist() if token != -100], skip_special_tokens=True
    )
    print(f"Example {idx}:")
    print("Assistant Response:", assistant_response)
    print("Decoded Labels:", decoded_labels)
    print("Match:", assistant_response.strip() == decoded_labels.strip())

Processed batch keys: KeysView({'pixel_values': tensor([[[[[-0.8039, -0.8039, -0.8039,  ..., -0.6471, -0.6392, -0.6392],
           [-0.8039, -0.8039, -0.8039,  ..., -0.6471, -0.6392, -0.6392],
           [-0.8039, -0.8039, -0.8039,  ..., -0.6471, -0.6392, -0.6392],
           ...,
           [ 0.1216,  0.1137,  0.1137,  ...,  0.6471,  0.6471,  0.6392],
           [ 0.1216,  0.1137,  0.1137,  ...,  0.6549,  0.6549,  0.6471],
           [ 0.1137,  0.1137,  0.1137,  ...,  0.6549,  0.6549,  0.6549]],

          [[-0.8667, -0.8667, -0.8667,  ..., -0.6627, -0.6549, -0.6471],
           [-0.8667, -0.8667, -0.8667,  ..., -0.6627, -0.6549, -0.6471],
           [-0.8667, -0.8667, -0.8667,  ..., -0.6627, -0.6549, -0.6471],
           ...,
           [-0.4353, -0.4353, -0.4353,  ...,  0.2235,  0.2314,  0.2314],
           [-0.4353, -0.4353, -0.4353,  ...,  0.2314,  0.2314,  0.2314],
           [-0.4353, -0.4353, -0.4353,  ...,  0.2314,  0.2314,  0.2314]],

          [[-0.9529, -0.9529, -0.9529,  

In [ ]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): Idefics3ForConditionalGeneration(
          (model): Idefics3Model(
            (vision_model): Idefics3VisionTransformer(
              (embeddings): Idefics3VisionEmbeddings(
                (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
                (position_embedding): Embedding(1024, 768)
              )
              (encoder): Idefics3Encoder(
                (layers): ModuleList(
                  (0-11): 12 x Idefics3EncoderLayer(
                    (self_attn): Idefics3VisionAttention(
                      (k_proj): lora.Linear(
                        (base_layer): Linear(in_features=768, out_features=768, bias=True)
                        (lora_dropout): ModuleDict(
                          (default): Dropout(p=0.05, inplace=False)
                        )
                        (lora_A): Modul

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=32,                 # 16–64 work; 32 is a good balance
    lora_alpha=32,        # usually = r or 2*r
    lora_dropout=0.05,    # 0.05–0.1
    use_rslora=True,      # RS-LoRA helps stability
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        # text
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
        # vision attention
        "q_proj","k_proj","v_proj","out_proj",
        # connector
        "modality_projection.proj",
        # optional, if VRAM allows:
        # "fc1","fc2",
    ],
    modules_to_save=["lm_head","embed_tokens"]   # keep head/tok if you changed them
)


In [ ]:
from peft import get_peft_model

model=get_peft_model(model,lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
model.print_trainable_parameters()

trainable params: 114,767,872 || all params: 622,250,176 || trainable%: 18.4440


In [ ]:
from google.colab import drive
drive.mount(f'{WORK_DIR}/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1) Choose a folder in Drive for checkpoints & logs
RUN_NAME = "smolvlm-lora-2e-4-e2"
CKPT_DIR = f"{RUNS_DIR}/{RUN_NAME}"
LOG_DIR  = f"{RUNS_DIR}/{RUN_NAME}/logs"

# 2) TrainingArguments that save to Drive
from transformers import TrainingArguments

training_args = TrainingArguments(
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="constant",
    warmup_ratio=0.03,

    logging_steps=200,
    eval_strategy="steps",    # <-- correct name
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    bf16=True,
    tf32=True,

    output_dir=CKPT_DIR,            # <-- saves to Google Drive
    remove_unused_columns=False,
    report_to="tensorboard",
    run_name=RUN_NAME,
    logging_dir=LOG_DIR,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
)

In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=3,   # stop if val loss doesn’t improve for 3 evals
        early_stopping_threshold=0.0 # how much improvement counts
    )],
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


# Training and pushing to the hub

We have all the core building blocks now, so we fine-tune the model!

The training can take a few minutes depending on the hardware you use.

In [ ]:
batch = next(iter(trainer.get_train_dataloader()))
labels = batch["labels"]
print("Masked ratio:", (labels == -100).float().mean().item())


Masked ratio: 0.9043209552764893


In [ ]:
# set cache use to false
model.config.use_cache = False

trainer.train()

<IPython.core.display.HTML object>

TrainOutput(global_step=5894, training_loss=0.5441060617975676, metrics={'train_runtime': 48143.9979, 'train_samples_per_second': 0.49, 'train_steps_per_second': 0.122, 'total_flos': 8.262202063958093e+16, 'train_loss': 0.5441060617975676, 'epoch': 2.0})

In [ ]:
import json, os

run_dir = f"{RUNS_DIR}/smolvlm-lora-2e-4-e2"
latest_ckpt = os.path.join(run_dir, "checkpoint-5894")  # you can also try checkpoint-5800

state_path = os.path.join(latest_ckpt, "trainer_state.json")

if os.path.exists(state_path):
    with open(state_path) as f:
        state = json.load(f)
    print("✅ Found trainer_state.json in:", latest_ckpt)
    print("Best checkpoint according to Trainer:", state.get("best_model_checkpoint"))
else:
    print("❌ trainer_state.json not found in", latest_ckpt)


✅ Found trainer_state.json in: /content/drive/MyDrive/SmolVLM_runs/smolvlm-lora-2e-4-e2/checkpoint-5894
Best checkpoint according to Trainer: /content/drive/MyDrive/SmolVLM_runs/smolvlm-lora-2e-4-e2/checkpoint-5800


In [ ]:
import os, json
from safetensors.torch import load_file

ckpt_path = f"{RUNS_DIR}/smolvlm-lora-2e-4-e2/checkpoint-5800"

print("FILES:", os.listdir(ckpt_path))

# adapter_config.json tells us adapter name, peft type, maybe base id
cfg_path = os.path.join(ckpt_path, "adapter_config.json")
if os.path.exists(cfg_path):
    with open(cfg_path) as f:
        cfg = json.load(f)
    print("\nadapter_config.json:")
    for k, v in cfg.items():
        print(f"  {k}: {v}")

# show a few keys from the adapter weights
bin_path = os.path.join(ckpt_path, "adapter_model.safetensors")
if not os.path.exists(bin_path):
    bin_path = os.path.join(ckpt_path, "adapter_model.bin")

if os.path.exists(bin_path):
    sd = load_file(bin_path) if bin_path.endswith(".safetensors") else torch.load(bin_path, map_location="cpu")
    some_keys = list(sd.keys())[:30]
    print("\nFirst 30 keys in adapter weights:")
    for k in some_keys:
        print(" ", k)
else:
    print("No adapter weights file found.")


FILES: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth', 'trainer_state.json']

adapter_config.json:
  alpha_pattern: {}
  auto_mapping: None
  base_model_name_or_path: None
  bias: none
  corda_config: None
  eva_config: None
  exclude_modules: None
  fan_in_fan_out: False
  inference_mode: True
  init_lora_weights: True
  layer_replication: None
  layers_pattern: None
  layers_to_transform: None
  loftq_config: {}
  lora_alpha: 32
  lora_bias: False
  lora_dropout: 0.05
  megatron_config: None
  megatron_core: megatron.core
  modules_to_save: ['lm_head', 'embed_tokens']
  peft_type: LORA
  qalora_group_size: 16
  r: 32
  rank_pattern: {}
  revision: None
  target_modules: ['up_proj', 'down_proj', 'out_proj', 'q_proj', 'gate_proj', 'v_proj', 'modality_projection.proj', 'k_proj', 'o_proj']
  target_parameters: None
  task_type: CAUSAL_LM
  trainable_token_indices: None
  use_dora: False
  use_qalora: 

In [ ]:
import os, torch, json, re
from safetensors.torch import load_file, save_file
from peft import PeftModel
from transformers import Idefics3ForConditionalGeneration, AutoProcessor

ckpt_path = f"{RUNS_DIR}/smolvlm-lora-2e-4-e2/checkpoint-5800"
fixed_path = ckpt_path + "_fixed"

os.makedirs(fixed_path, exist_ok=True)

# 1) copy adapter_config.json as-is
import shutil, glob
shutil.copy2(os.path.join(ckpt_path, "adapter_config.json"),
             os.path.join(fixed_path, "adapter_config.json"))

# 2) load, rename keys, save
src = os.path.join(ckpt_path, "adapter_model.safetensors")
dst = os.path.join(fixed_path, "adapter_model.safetensors")
sd = load_file(src)

def fix_key(k: str) -> str:
    # collapse multiple "base_model.model." into one
    # e.g., "base_model.model.base_model.model.base_model.model." -> "base_model.model."
    while "base_model.model.base_model.model." in k:
        k = k.replace("base_model.model.base_model.model.", "base_model.model.")
    return k

new_sd = {fix_key(k): v for k, v in sd.items()}
save_file(new_sd, dst)

print("Wrote fixed adapter:", dst)


Wrote fixed adapter: /content/drive/MyDrive/SmolVLM_runs/smolvlm-lora-2e-4-e2/checkpoint-5800_fixed/adapter_model.safetensors


In [ ]:
import torch, os
from transformers import Idefics3ForConditionalGeneration, AutoProcessor
from peft import PeftModel

fixed_path = f"{RUNS_DIR}/smolvlm-lora-2e-4-e2/checkpoint-5800_fixed"

# pick the exact base model you trained with
base_candidates = [
    "HuggingFaceTB/SmolVLM-500M",
    "HuggingFaceTB/SmolVLM-500M-Instruct",
]

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

base_model, processor = None, None
last_err = None
for base_model_id in base_candidates:
    try:
        print(f"Trying base: {base_model_id}")
        base_model = Idefics3ForConditionalGeneration.from_pretrained(
            base_model_id, torch_dtype=torch_dtype, device_map="auto"
        )
        processor = AutoProcessor.from_pretrained(base_model_id)
        print("Base loaded OK:", base_model_id)

        # attach adapter (adapter name is 'base_model' per your keys)
        model = PeftModel.from_pretrained(base_model, fixed_path, adapter_name="base_model")
        model.eval()
        print("LoRA attached OK with adapter_name='base_model'")
        chosen_base = base_model_id
        break
    except Exception as e:
        last_err = e
        print("Failed with:", base_model_id, "->", repr(e))

if model is None:
    raise RuntimeError(f"Could not load base + adapter with any candidate. Last error: {last_err}")


Trying base: HuggingFaceTB/SmolVLM-500M
Failed with: HuggingFaceTB/SmolVLM-500M -> OSError("HuggingFaceTB/SmolVLM-500M is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'\nIf this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`")
Trying base: HuggingFaceTB/SmolVLM-500M-Instruct
Base loaded OK: HuggingFaceTB/SmolVLM-500M-Instruct
LoRA attached OK with adapter_name='base_model'


In [ ]:
from PIL import Image
import torch

def smolvlm_caption(image: Image.Image, question="What do you see here?"):
    # Chat message with one image
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text",  "text": question},
                {"type": "image"}  # the single image in `images=` below
            ],
        }
    ]

    # 1) Get the prompt as a STRING (not tensors)
    prompt_str = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False  # <-- important
    )

    # 2) Build tensors from the string + image
    batch = processor(
        text=[prompt_str],             # wrap in list for batch size 1
        images=[image],                # wrap in list for batch size 1
        return_tensors="pt",
    )

    # 3) Move to device in one go
    batch = {k: v.to(model.device) for k, v in batch.items()}

    with torch.no_grad():
        gen_ids = model.generate(
            **batch,
            max_new_tokens=128,
            num_beams=3,
            do_sample=False,
            early_stopping=True,
        )

    text = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return text


In [ ]:
test_img = Image.open(f"{CAPTIONS_DIR}/Test-image/0012821d6f11b96cf33f2c2ee5c68d1f.jpg").convert("RGB")
print(smolvlm_caption(test_img))


User: What do you see here?




Assistant: The image shows a close-up of a person's eye;  revealing a cluster of flesh-colored lid papules. These lesions are symmetrically distributed on the eyelids;  appearing larger than normal. The surface of the lesions is smooth and unremarkable;  without any scaling;  crust;  or ulceration. The surrounding skin appears normal;  without any visible changes.


In [ ]:
import os, csv
from PIL import Image

test_dir = f"{CAPTIONS_DIR}/Test-image"  # change me
out_csv  = f"{WORK_DIR}/test_sample_10_smolvlm.csv"

valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
files = [f for f in sorted(os.listdir(test_dir)) if os.path.splitext(f)[1].lower() in valid_ext][:10]

rows = []
for f in files:
    try:
        img = Image.open(os.path.join(test_dir, f)).convert("RGB")
        cap = smolvlm_caption(img)
        print(f"{f} -> {cap[:100]}...")
        rows.append({"image": f, "caption": cap})
    except Exception as e:
        print("Failed on", f, ":", repr(e))

with open(out_csv, "w", newline="", encoding="utf-8") as fp:
    w = csv.DictWriter(fp, fieldnames=["image", "caption"])
    w.writeheader()
    w.writerows(rows)

print("Saved:", out_csv)


0012821d6f11b96cf33f2c2ee5c68d1f.jpg -> User: What do you see here?




Assistant: The image shows a close-up of a person's eye;  revealing ...
001d22ff2543f95d2d38c18da0446c84.jpg -> User: What do you see here?




Assistant: The image shows a skin lesion with a reddish-brown color;...
002714e65a78f16fb05bc0aa95ea9761.jpg -> User: What do you see here?




Assistant: The image shows a chronic inflammatory skin condition;  s...
003e6abf20d234221a41b528b946e90c.jpg -> User: What do you see here?



Assistant: The image shows a person's arm with chronic itchy skin;  c...
005b804472c7a27908f99e3d6d6cf91c.jpg -> User: What do you see here?





Assistant: The image shows a skin lesion with a blisters and scalin...
0060c63deaf2c1a022136b1f6b04b87d.jpg -> User: What do you see here?




Assistant: The image shows a close-up of a person's fingers;  with a...
00918f1c84f591a61d5e59600db86f05.jpg -> User: What do you see here?





Assistant: The image depicts a cutaneous squamous cell carcinom

In [ ]:
from peft import PeftModel

def show_adapter_status(m):
    # Is this a PEFT model?
    print("Is PeftModel:", isinstance(m, PeftModel))
    # What adapters exist in the model?
    if hasattr(m, "peft_config"):
        print("Adapters found:", list(m.peft_config.keys()))
    # Which adapter is active?
    if hasattr(m, "active_adapter"):
        print("Active adapter:", m.active_adapter)
    # Quick sanity on LoRA tensors actually present
    lora_tensors = [k for k in m.state_dict().keys() if "lora_A" in k or "lora_B" in k]
    print("Number of LoRA tensors in state_dict:", len(lora_tensors))

show_adapter_status(model)


Is PeftModel: True
Adapters found: ['base_model']
Active adapter: base_model
Number of LoRA tensors in state_dict: 546


TEST PREDICTION

In [ ]:
# --- CONFIG -------------------------------------------------------------------
BASE_ID     = "HuggingFaceTB/SmolVLM-500M-Instruct"   # base SmolVLM
ADAPTER_DIR = f"{RUNS_DIR}/smolvlm-lora-2e-4-e2/checkpoint-5800_fixed"  # <- your adapter
TEST_DIR    = f"{CAPTIONS_DIR}/Test-image"
OUT_CSV     = f"{RUNS_DIR}/newtest_inference_smolvlm_lora.csv"
DEVICE      = "cuda"  # or "mps"/"cpu" if needed
QUESTION    = "What do you see here?"  # prompt used during fine-tuning (keep it consistent)

# --- LOAD MODEL + ADAPTER -----------------------------------------------------
import os, json, torch
from PIL import Image
from tqdm import tqdm
import pandas as pd

from transformers import AutoProcessor, Idefics3ForConditionalGeneration
from peft import PeftModel

torch.set_grad_enabled(False)

print("Loading base model …")
processor = AutoProcessor.from_pretrained(BASE_ID)
model = Idefics3ForConditionalGeneration.from_pretrained(
    BASE_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",  # puts model on GPU if available
)

print(f"Attaching LoRA from: {ADAPTER_DIR}")
model = PeftModel.from_pretrained(model, ADAPTER_DIR, is_trainable=False)
model.eval()

# --- Sanity check: adapter really active --------------------------------------
def show_adapter_status(m):
    from peft import PeftModel
    print("Is PeftModel:", isinstance(m, PeftModel))
    if hasattr(m, "peft_config"):
        print("Adapters found:", list(m.peft_config.keys()))
    if hasattr(m, "active_adapter"):
        print("Active adapter:", m.active_adapter)
    lora_tensors = [k for k in m.state_dict().keys() if "lora_A" in k or "lora_B" in k]
    print("Number of LoRA tensors in state_dict:", len(lora_tensors))

show_adapter_status(model)

# --- Caption helper (SmolVLM chat template style) -----------------------------
def generate_caption(image: Image.Image, question: str = QUESTION) -> str:
    """
    Generates a caption for a single PIL image using the active LoRA adapter.
    """
    # SmolVLM expects a chat-style message with an <image> turn
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text",  "text": question},
                {"type": "image"}    # image comes after the text
            ],
        }
    ]

    # Build input text using the model's conversation template
    text = processor.apply_chat_template(messages, add_generation_prompt=True)

    # Tokenize (text + image) and move to device
    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    )
    inputs = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

    # Generate
    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,         # beam search is usually more stable for captioning
            num_beams=3,
            early_stopping=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    # Decode *only* the newly generated tokens (skip the prompt)
    prompt_len = inputs["input_ids"].shape[-1]
    out = processor.tokenizer.batch_decode(gen_ids[:, prompt_len:], skip_special_tokens=True)[0]
    return out.strip()

# --- Gather test images --------------------------------------------------------
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def list_images(root):
    files = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if os.path.splitext(f.lower())[1] in IMG_EXTS:
                files.append(os.path.join(dirpath, f))
    files.sort()
    return files

test_images = list_images(TEST_DIR)
print(f"Found {len(test_images)} images in {TEST_DIR}")

# --- Run inference over the entire test set -----------------------------------
rows = []
for img_path in tqdm(test_images, desc="Captioning"):
    try:
        img = Image.open(img_path).convert("RGB")
        cap = generate_caption(img)
    except Exception as e:
        cap = f"[ERROR] {e}"
    rows.append({"image": os.path.relpath(img_path, TEST_DIR), "caption": cap})

# --- Save results --------------------------------------------------------------
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print(f"✅ Saved captions to: {OUT_CSV}")


Loading base model …
Attaching LoRA from: /content/drive/MyDrive/SmolVLM_runs/smolvlm-lora-2e-4-e2/checkpoint-5800_fixed
Is PeftModel: True
Adapters found: ['default']
Active adapter: default
Number of LoRA tensors in state_dict: 546
Found 3316 images in /content/drive/MyDrive/Skin_Concepts/Skin_Concepts_datasets/new captions_qwen_rag/Test-image


Captioning: 100%|██████████| 3316/3316 [8:55:51<00:00,  9.70s/it]

✅ Saved captions to: /content/drive/MyDrive/SmolVLM_runs/newtest_inference_smolvlm_lora.csv


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os
import textwrap

# === PATHS ===
CSV_PATH = f"{RUNS_DIR}/newtest_inference_smolvlm_lora.csv"
TEST_DIR = f"{CAPTIONS_DIR}/Test-image"

# === LOAD CSV ===
df = pd.read_csv(CSV_PATH)
print("Total captions:", len(df))
sample_df = df.head(10)  # first 10 samples

# === PLOT SETTINGS ===
plt.figure(figsize=(20, 16))  # wider figure
cols = 2
rows = int(len(sample_df) / cols) + (len(sample_df) % cols > 0)

for i, row in enumerate(sample_df.itertuples(), 1):
    img_path = os.path.join(TEST_DIR, row.image)
    try:
        img = Image.open(img_path).convert("RGB")
    except:
        continue

    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.axis("off")

    # wrap caption text for readability
    wrapped = "\n".join(textwrap.wrap(row.caption, width=80))
    plt.title(f"{i}. {wrapped}", fontsize=11, pad=10)

plt.tight_layout()
plt.show()



Total captions: 3316


<Figure size 2000x1600 with 10 Axes>

# Evaluation

Let's evaluate the model. First, we can have a look at a qualitative generation from the model.

In [ ]:
eval_results = run_model_evaluation(model, eval_dataset, num_samples=3, device='cuda',
                                    constant_query="What do you see here?"
                                    # constant_query="Que voyez vous ici?"
                                   )

<PIL.Image.Image image mode=RGB size=300x400>

Formatted prompt: <|im_start|>User: What do you see here?<image><end_of_utterance>
Assistant:
Prediction:  A white queen and a white rook.



<PIL.Image.Image image mode=RGB size=300x400>

Formatted prompt: <|im_start|>User: What do you see here?<image><end_of_utterance>
Assistant:
Prediction:  A white queen.



<PIL.Image.Image image mode=RGB size=300x400>

Formatted prompt: <|im_start|>User: What do you see here?<image><end_of_utterance>
Assistant:
Prediction:  A white rook, a black knight and a white pawn.



# Push to Hub

In [ ]:
# Merge the model
model = model.merge_and_unload()

In [ ]:
# Technically, if you want the best performance for a quantized model, you should save and push your lora adapters, reload the base model, then dequantize that base model, then reload the adapter on top of that, then merge...
# model = model.dequantize() # may be a useful command here.

TEST CAPTION PREDICTED +GROUND TRUTH

In [ ]:
import pandas as pd
import re
import os

# ====== Paths to your files ======
pred_path = f"{RUNS_DIR}/Smolvlm_Caption.csv"   # predicted captions
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"  # ground truth




# ====== Load both files ======
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

print("Predicted CSV columns:", list(pred_df.columns))
print("Ground Truth CSV columns:", list(gt_df.columns))

# ====== Normalize column names ======
pred_df.columns = pred_df.columns.str.strip().str.lower()
gt_df.columns   = gt_df.columns.str.strip().str.lower()

# Ensure we have 'id' and 'caption'
pred_df = pred_df.rename(columns={"caption": "caption_pred", "id": "id_pred"})
gt_df   = gt_df.rename(columns={"caption": "caption_true", "id": "id_true"})

# ====== Clean IDs: remove paths and extensions ======
def clean_id(x):
    x = str(x)
    x = os.path.basename(x)  # remove path
    x = re.sub(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif|webp)$", "", x, flags=re.I)  # remove extension
    return x.lower().strip()

pred_df["key"] = pred_df["id_pred"].apply(clean_id)
gt_df["key"]   = gt_df["id_true"].apply(clean_id)

# ====== Merge on cleaned keys ======
merged = pd.merge(pred_df, gt_df, on="key", how="inner")
print(f"Merged rows: {len(merged)}")

# ====== Display first few samples ======
from IPython.display import display

sample = merged[["key", "caption_pred", "caption_true"]].head(10)
display(sample)

# ====== Optional: text print ======
for _, row in sample.iterrows():
    print(f"\n🧩 ID: {row['key']}")
    print(f"🔵 Predicted Caption: {row['caption_pred']}")
    print(f"🟢 Ground Truth Caption: {row['caption_true']}")
    print("-" * 100)



Predicted CSV columns: ['ID', 'Caption']
Ground Truth CSV columns: ['ID', 'Caption']
Merged rows: 3316


                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   
3  003e6abf20d234221a41b528b946e90c   
4  005b804472c7a27908f99e3d6d6cf91c   
5  0060c63deaf2c1a022136b1f6b04b87d   
6  00918f1c84f591a61d5e59600db86f05   
7  0094c6373a5ac2add4f2b8bc78571243   
8  00a61ae0aa6d43a08152a7c4692ef9e2   
9  00c6529c53944cfc7b184abce798e16a   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a skin lesion with a reddish-b...   
2  The image shows a chronic inflammatory skin co...   
3  The image shows a person's arm with chronic it...   
4  The image shows a skin lesion with a blisters ...   
5  The image shows a close-up of a person's finge...   
6  The image depicts a cutaneous squamous cell ca...   
7  The image depicts a cutaneous squamous cell ca...   
8  The image shows a person's arm with a reddish-...   
9  The ima


🧩 ID: 0012821d6f11b96cf33f2c2ee5c68d1f
🔵 Predicted Caption: The image shows a close-up of a person's eye;  revealing a cluster of flesh-colored lid papules. These lesions are symmetrically distributed on the eyelids;  appearing larger than normal. The surface of the lesions is smooth and unremarkable;  without any scaling;  crust;  or ulceration. The surrounding skin appears normal;  without any visible changes.
🟢 Ground Truth Caption: The image shows a close-up of a person's eye;  revealing a cluster of flesh-colored lid papules. These lesions are symmetrically distributed and appear larger than normal. The surface of the papules is smooth and unremarkable. There are no surrounding skin changes or notable features.
----------------------------------------------------------------------------------------------------

🧩 ID: 001d22ff2543f95d2d38c18da0446c84
🔵 Predicted Caption: The image shows a skin lesion with a reddish-brown color;  located on the lower abdomen. The lesion appears to 

In [ ]:
# === Install (if needed)
!pip -q install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00


BIOCLINICAL BERTSCORE

In [ ]:
# === Install (if needed)
!pip -q install bert-score

import pandas as pd, re, os, torch
from bert_score import score
from IPython.display import display

# ---- Paths (your files)
pred_path = f"{RUNS_DIR}/Smolvlm_Caption.csv"  # predicted captions (ID, Caption)
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"  # ground truth (ID, Caption)

# ---- Helpers
def clean_id(x: str) -> str:
    x = os.path.basename(str(x))
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|webp|tif|tiff)$", "", x, flags=re.I)  # drop extension
    return x.strip().lower()

# ---- Load
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

# Normalize columns
pred_df.columns = pred_df.columns.str.strip().str.lower()
gt_df.columns   = gt_df.columns.str.strip().str.lower()
pred_df = pred_df.rename(columns={"id":"id_pred", "caption":"caption_pred"})
gt_df   = gt_df.rename(columns={"id":"id_true", "caption":"caption_true"})

# Build merge key (drop .jpg etc.)
pred_df["key"] = pred_df["id_pred"].apply(clean_id)
gt_df["key"]   = gt_df["id_true"].apply(clean_id)

# Merge
merged = pred_df.merge(gt_df[["key","caption_true"]], on="key", how="inner")
print("Merged rows:", len(merged))
display(merged[["key","caption_pred","caption_true"]].head(5))

# Lists for BERTScore
preds = merged["caption_pred"].astype(str).tolist()
refs  = merged["caption_true"].astype(str).tolist()

# ---- BERTScore (RoBERTa-large, baseline) — ImageCLEF-style
try:
    P, R, F1 = score(
        preds, refs,
        model_type="roberta-large",
        lang="en",
        rescale_with_baseline=True,
        idf=True,  # if your version errors, fall back below
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
except TypeError:
    # older bert-score versions use `use_idf`
    P, R, F1 = score(
        preds, refs,
        model_type="roberta-large",
        lang="en",
        rescale_with_baseline=True,
        use_idf=True,
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

print(f"\nRoBERTa-large (baseline + IDF) — Avg BERTScore F1: {float(F1.mean()):.4f}")

# ---- Optional: domain variant for reference (BioClinicalBERT, no baseline)
try:
    P2, R2, F12 = score(
        preds, refs,
        model_type="emilyalsentzer/Bio_ClinicalBERT",
        num_layers=12,           # required for non-listed model names
        lang=None,
        rescale_with_baseline=False,
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    print(f"BioClinicalBERT (no baseline) — Avg BERTScore F1: {float(F12.mean()):.4f}")
except Exception as e:
    print("BioClinicalBERT variant skipped:", e)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
Merged rows: 3316


                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   
3  003e6abf20d234221a41b528b946e90c   
4  005b804472c7a27908f99e3d6d6cf91c   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a skin lesion with a reddish-b...   
2  The image shows a chronic inflammatory skin co...   
3  The image shows a person's arm with chronic it...   
4  The image shows a skin lesion with a blisters ...   

                                        caption_true  
0  The image shows a close-up of a person's eye; ...  
1  The image shows a person's skin with a cluster...  
2  The lesion is flesh-colored and appears to be ...  
3  The image shows a leg with a visible lesion. T...  
4  The image shows a skin lesion with a reddish-b...  

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BioClinicalBERT (no baseline) — Avg BERTScore F1: 0.8206


In [ ]:
import re
import torch
from bert_score import score

# Pre-processing function
def preprocess_caption(s: str) -> str:
    s2 = str(s).lower()
    # replace numbers with token “number”
    s2 = re.sub(r"\d+", "number", s2)
    # remove punctuation (everything that is not alphanumeric or space)
    s2 = re.sub(r"[^\w\s]", "", s2)
    s2 = re.sub(r"\s+", " ", s2).strip()
    return s2

# Make lists
preds_pp = [preprocess_caption(x) for x in preds]  # your predicted captions
refs_pp  = [preprocess_caption(x) for x in refs]   # ground truth captions

# Compute BERTScore (Recall + IDF) — use model you have; default “roberta-large” works
P, R, F1 = score(
    preds_pp, refs_pp,
    model_type="roberta-large",
    lang="en",
    rescale_with_baseline=False,  # Using recall-focused; baseline sometimes off
    idf=True,
    batch_size=16,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("ImageCLEF-style BERTScore (Recall+IDF) — avg Recall:", float(R.mean()))


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ImageCLEF-style BERTScore (Recall+IDF) — avg Recall: 0.8416600823402405


BERTSCORE

In [ ]:
!pip install -q bert-score

import pandas as pd
import re, os, torch
from bert_score import score
from IPython.display import display

# ====== CSV paths ======
pred_path = f"{RUNS_DIR}/Smolvlm_Caption.csv"  # predicted captions
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"  # ground truth

# ====== Load CSVs ======
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

# Normalize column names
pred_df.columns = pred_df.columns.str.strip().str.lower()
gt_df.columns   = gt_df.columns.str.strip().str.lower()

# Rename
pred_df = pred_df.rename(columns={"id": "id_pred", "caption": "caption_pred"})
gt_df   = gt_df.rename(columns={"id": "id_true", "caption": "caption_true"})

# ====== Clean IDs (drop .jpg, .png, etc.) ======
def clean_id(x):
    x = str(x)
    x = os.path.basename(x)
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.lower().strip()

pred_df["key"] = pred_df["id_pred"].apply(clean_id)
gt_df["key"]   = gt_df["id_true"].apply(clean_id)

# ====== Merge ======
merged = pred_df.merge(gt_df[["key", "caption_true"]], on="key", how="inner")
print(f"Merged rows: {len(merged)}")
display(merged.head(5))

# ====== ImageCLEF-style preprocessing ======
def preprocess_caption(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

preds = [preprocess_caption(x) for x in merged["caption_pred"]]
refs  = [preprocess_caption(x) for x in merged["caption_true"]]

# ====== Compute BERTScore (ImageCLEF-style) ======
P, R, F1 = score(
    preds, refs,
    model_type="roberta-large",
    lang="en",
    rescale_with_baseline=False,  # ImageCLEF uses raw recall with IDF
    idf=True,                     # IDF weighting on test corpus
    batch_size=16,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("\n=== ImageCLEF-style BERTScore ===")
print(f"Precision: {float(P.mean()):.4f}")
print(f"Recall:    {float(R.mean()):.4f}  <-- primary metric used by ImageCLEF")
print(f"F1:        {float(F1.mean()):.4f}")

# ====== Save merged captions (optional) ======
out_csv = f"{RUNS_DIR}/merged_for_bertscore.csv"
merged.to_csv(out_csv, index=False)
print(f"\nMerged data saved to: {out_csv}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.1 MB/s eta 0:00:00
Merged rows: 3316


                                id_pred  \
0  0012821d6f11b96cf33f2c2ee5c68d1f.jpg   
1  001d22ff2543f95d2d38c18da0446c84.jpg   
2  002714e65a78f16fb05bc0aa95ea9761.jpg   
3  003e6abf20d234221a41b528b946e90c.jpg   
4  005b804472c7a27908f99e3d6d6cf91c.jpg   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a skin lesion with a reddish-b...   
2  The image shows a chronic inflammatory skin co...   
3  The image shows a person's arm with chronic it...   
4  The image shows a skin lesion with a blisters ...   

                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   
3  003e6abf20d234221a41b528b946e90c   
4  005b804472c7a27908f99e3d6d6cf91c   

                                        caption_true  
0  The image shows a close-up of a person's eye; ...  
1  The image shows a person's skin with a cluster...  
2  T

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== ImageCLEF-style BERTScore ===
Precision: 0.8417
Recall:    0.8417  <-- primary metric used by ImageCLEF
F1:        0.8415

Merged data saved to: /content/drive/MyDrive/SmolVLM_runs/merged_for_bertscore.csv


ROGUE

In [ ]:
!pip -q install rouge-score

import pandas as pd, re, os
from rouge_score import rouge_scorer
from IPython.display import display

# ====== CSV paths (same as before) ======
pred_path = f"{RUNS_DIR}/Smolvlm_Caption.csv"  # predicted captions (ID, Caption)
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"  # ground truth (ID, Caption)

# ====== Helpers ======
def clean_id(x: str) -> str:
    x = os.path.basename(str(x))
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.lower().strip()

def preprocess_caption(s: str) -> str:
    # ImageCLEF-style: lowercase, digits→'number', strip punctuation, squeeze spaces
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ====== Load & merge ======
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

pred_df.columns = pred_df.columns.str.strip().str.lower()
gt_df.columns   = gt_df.columns.str.strip().str.lower()
pred_df = pred_df.rename(columns={"id":"id_pred","caption":"caption_pred"})
gt_df   = gt_df.rename(columns={"id":"id_true","caption":"caption_true"})

pred_df["key"] = pred_df["id_pred"].apply(clean_id)
gt_df["key"]   = gt_df["id_true"].apply(clean_id)

merged = pred_df.merge(gt_df[["key","caption_true"]], on="key", how="inner")
print("Merged rows:", len(merged))
display(merged.head(3)[["key","caption_pred","caption_true"]])

# ====== Preprocess captions ======
preds = [preprocess_caption(x) for x in merged["caption_pred"]]
refs  = [preprocess_caption(x) for x in merged["caption_true"]]

# ====== ROUGE scorer (with stemming as commonly used) ======
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

r1_r, r1_f, r2_r, r2_f, rl_r, rl_f = [], [], [], [], [], []
for pred, ref in zip(preds, refs):
    s = scorer.score(ref, pred)   # (reference, prediction)
    r1_r.append(s['rouge1'].recall);  r1_f.append(s['rouge1'].fmeasure)
    r2_r.append(s['rouge2'].recall);  r2_f.append(s['rouge2'].fmeasure)
    rl_r.append(s['rougeL'].recall);  rl_f.append(s['rougeL'].fmeasure)

print("\n=== ImageCLEF-style ROUGE ===")
print(f"ROUGE-1  Recall: {sum(r1_r)/len(r1_r):.4f}   F1: {sum(r1_f)/len(r1_f):.4f}")
print(f"ROUGE-2  Recall: {sum(r2_r)/len(r2_r):.4f}   F1: {sum(r2_f)/len(r2_f):.4f}")
print(f"ROUGE-L  Recall: {sum(rl_r)/len(rl_r):.4f}   F1: {sum(rl_f)/len(rl_f):.4f}   <-- commonly reported")

# (optional) attach per-sample ROUGE to your dataframe and save
merged = merged.copy()
merged["rouge1_f"] = r1_f; merged["rouge2_f"] = r2_f; merged["rougeL_f"] = rl_f
out_csv = f"{RUNS_DIR}/merged_with_imageclef_rouge.csv"
merged.to_csv(out_csv, index=False)
print(f"\nSaved per-sample ROUGE to: {out_csv}")


  Preparing metadata (setup.py) ... done
Merged rows: 3316


                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a skin lesion with a reddish-b...   
2  The image shows a chronic inflammatory skin co...   

                                        caption_true  
0  The image shows a close-up of a person's eye; ...  
1  The image shows a person's skin with a cluster...  
2  The lesion is flesh-colored and appears to be ...  


=== ImageCLEF-style ROUGE ===
ROUGE-1  Recall: 0.5255   F1: 0.5183
ROUGE-2  Recall: 0.2642   F1: 0.2608
ROUGE-L  Recall: 0.3806   F1: 0.3755   <-- commonly reported

Saved per-sample ROUGE to: /content/drive/MyDrive/SmolVLM_runs/merged_with_imageclef_rouge.csv


In [ ]:
image_path=f"{CAPTIONS_DIR}/Test-image"

TEST IMAGES AND KEYWORDS

In [ ]:
import pandas as pd
import json

# ==== EDIT THESE ====
TEST_PATH  = f"{WORK_DIR}/test - test.csv.csv"  # has: image_id, label
FINAL_PATH = f"{WORK_DIR}/final_image_label_concepts - final_image_label_concepts.csv (1).csv"  # has: label, concepts
OUT_PATH   = f"{WORK_DIR}//test_with_keywords.csv"

# ---- load & normalize column names ----
test_df  = pd.read_csv(TEST_PATH)
final_df = pd.read_csv(FINAL_PATH)
test_df.columns  = test_df.columns.str.strip().str.lower()
final_df.columns = final_df.columns.str.strip().str.lower()

# sanity: required columns
for df, need in [(test_df, {"image_id","label"}), (final_df, {"label","concepts"})]:
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns {missing} in dataframe.")

# ---- make sure 'concepts' is a list per row, then aggregate per label ----
def to_list(x):
    # accept Python-list-like strings or comma-separated strings
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    try:
        v = json.loads(s)            # handles "['a','b']" or '["a","b"]'
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        return [t.strip() for t in s.split(",") if t.strip()]

final_df["concepts_list"] = final_df["concepts"].apply(to_list)

# aggregate UNIQUE concepts per label
concepts_by_label = (
    final_df.groupby("label")["concepts_list"]
    .apply(lambda rows: sorted(set([c for lst in rows for c in lst])))
    .reset_index()
    .rename(columns={"concepts_list": "keywords"})
)

# ---- join onto test set by label ----
out = (
    test_df[["image_id","label"]]
    .merge(concepts_by_label, on="label", how="left")
    .rename(columns={"image_id": "ID"})
)

# convert keywords lists to JSON-like strings for CSV readability
out["keywords"] = out["keywords"].apply(lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, list) else "[]")

# ---- save ----
out.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")
print(out.head(5))


Saved: /content//test_with_keywords.csv
                                 ID               label  \
0  d395430d11d4ac72e6f60360aabf0e61  pustular psoriasis   
1  8dffbb994ef17963995f8059a76418b9        folliculitis   
2  01be7f7454385c1abaa9d10aabcaa751       fordyce spots   
3  e1e0b7f3462e4d9c5819e81c22d8238d      scleromyxedema   
4  c7fcb5f49fbe7fb7eeec7eaf196b299a             scabies   

                                            keywords  
0  ["'erythematous skin'", "'painful skin'", "'pu...  
1  ["'acne'", "'bacterial folliculitis.']", "'dee...  
2  ["'Fordyce granules.']", "'genital mucosa'", "...  
3  ["'generalised skin disorder'", "'lichen myxoe...  
4  ["'characteristic appearance'", "'distribution...  


KEYWORD MATCH

In [ ]:
# ===== Install if needed
!pip -q install bert-score

import pandas as pd, re, os, torch
from bert_score import score
from IPython.display import display

# ===== Paths (EDIT these if your files are elsewhere)
pred_path     = f"{RUNS_DIR}/Smolvlm_Caption.csv"     # has: ID, Caption  (predicted)
keywords_path = f"{WORK_DIR}/test_with_keywords.csv"        # has: ID, keywords

# ===== Helpers
def clean_id(x:str)->str:
    x = os.path.basename(str(x))
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.strip().lower()

def ic_preproc(s:str)->str:
    """ImageCLEF-style: lowercase, digits->'number', remove punctuation, squeeze spaces."""
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_simple(s:str):
    return set(re.findall(r"[a-z0-9]+", ic_preproc(s)))

# ===== Load
pred_df = pd.read_csv(pred_path)
kw_df   = pd.read_csv(keywords_path)

# Normalize column names
pred_df.columns = pred_df.columns.str.strip().str.lower()
kw_df.columns   = kw_df.columns.str.strip().str.lower()

# Rename to consistent names
pred_df = pred_df.rename(columns={"id":"id_pred", "caption":"caption_pred"})
if "keywords" not in kw_df.columns:
    raise ValueError("The keywords file must contain a 'keywords' column.")
kw_df = kw_df.rename(columns={"id":"id_kw"})

# Clean IDs (drop .jpg etc.)
pred_df["key"] = pred_df["id_pred"].apply(clean_id)
kw_df["key"]   = kw_df["id_kw"].apply(clean_id) if "id_kw" in kw_df.columns else kw_df["keywords"].index

# Merge
merged = pred_df.merge(kw_df[["key","keywords"]], on="key", how="inner")
print("Merged rows:", len(merged))
display(merged.head(3)[["key","caption_pred","keywords"]])

# ===== Build lists for scoring
preds_pp = [ic_preproc(x) for x in merged["caption_pred"].astype(str)]
keys_pp  = [ic_preproc(x) for x in merged["keywords"].astype(str)]

# ===== BERTScore (Recall) between caption (candidate) and keywords (reference)
# ImageCLEF uses RoBERTa-large; using recall+idf is most aligned with “keyword coverage”
try:
    P, R, F1 = score(
        preds_pp, keys_pp,
        model_type="roberta-large",
        lang="en",
        rescale_with_baseline=False,  # raw recall
        idf=True,                     # if your version errors, switch to use_idf=True
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
except TypeError:
    P, R, F1 = score(
        preds_pp, keys_pp,
        model_type="roberta-large",
        lang="en",
        rescale_with_baseline=False,
        use_idf=True,
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

merged["bertscore_recall_kw"] = R.tolist()
merged["bertscore_f1_kw"]     = F1.tolist()  # optional

# ===== Exact keyword-coverage (token overlap) — helpful sanity metric
cov_scores = []
for cap, kws in zip(merged["caption_pred"], merged["keywords"]):
    cap_tok = tokenize_simple(cap)
    kw_tok  = tokenize_simple(kws)
    if not kw_tok:
        cov_scores.append(0.0)
    else:
        cov_scores.append(len(cap_tok & kw_tok) / len(kw_tok))
merged["exact_keyword_coverage"] = cov_scores

# ===== Report
print("\n=== Caption vs Keywords ===")
print(f"BERTScore Recall (avg): {merged['bertscore_recall_kw'].mean():.4f}")
print(f"BERTScore F1     (avg): {merged['bertscore_f1_kw'].mean():.4f}")
print(f"Exact keyword coverage (avg): {merged['exact_keyword_coverage'].mean():.4f}")

# Show worst/best examples
print("\nLowest 5 by BERTScore Recall:")
display(merged.nsmallest(5, "bertscore_recall_kw")[["key","caption_pred","keywords","bertscore_recall_kw","exact_keyword_coverage"]])

print("\nHighest 5 by BERTScore Recall:")
display(merged.nlargest(5, "bertscore_recall_kw")[["key","caption_pred","keywords","bertscore_recall_kw","exact_keyword_coverage"]])

# ===== Save
out_csv = f"{RUNS_DIR}/caption_vs_keywords_bertscore.csv"
merged.to_csv(out_csv, index=False)
print(f"\nSaved per-sample metrics to: {out_csv}")


Merged rows: 3316


                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a skin lesion with a reddish-b...   
2  The image shows a chronic inflammatory skin co...   

                                            keywords  
0  ["'cheek papules'", "'clusters'", "'flesh-colo...  
1  ["'allergen'", "'allergic reaction'", "'contac...  
2  ["'cheek papules'", "'clusters'", "'flesh-colo...  

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...


=== Caption vs Keywords ===
BERTScore Recall (avg): 0.7960
BERTScore F1     (avg): 0.7909
Exact keyword coverage (avg): 0.1830

Lowest 5 by BERTScore Recall:


                                   key  \
3126  f17d44c2b7c221b55f8e079137ece231   
1714  856139b037b08042b0e650453aa92d5b   
2526  c3bbd1e08662fc2494e935e06e989b62   
2057  9f8781713471d95aa534f18c59494673   
903   45192b4792fbb24005537e2639fd139d   

                                           caption_pred  \
3126  The image shows a skin lesion with a reddish-b...   
1714  The image shows two knees with visible bruises...   
2526  The visible findings include a lesion on the l...   
2057  The image shows a close-up of a person's abdom...   
903   The image shows a close-up of a person's legs;...   

                                keywords  bertscore_recall_kw  \
3126  ["['Erythema chronicum migrans']"]             0.700211   
1714  ["['Erythema chronicum migrans']"]             0.703189   
2526  ["['Erythema chronicum migrans']"]             0.703753   
2057  ["['Erythema chronicum migrans']"]             0.706106   
903   ["['Erythema chronicum migrans']"]             0.707215   

 


Highest 5 by BERTScore Recall:


                                   key  \
513   26aa34a81a0772e63ed24236d93abe02   
947   48700506ed2b3d5b0e5b4cbc6a12ad94   
2165  a888959f2f6da0d96009edaa76a454da   
2304  b2665aed477aa85223b7fa2b9e72e368   
2818  da71631bf2076bf9eb937cad01ec06f9   

                                           caption_pred  \
513   The image shows a leg with a granulomatous ski...   
947   The image shows a scalp defect;  which is a co...   
2165  The image shows a scalp defect;  which is a co...   
2304  The image shows a scalp defect;  which is a co...   
2818  The image shows a scalp defect;  which is a co...   

                                               keywords  bertscore_recall_kw  \
513                 ["['granulomatous skin disorder']"]             0.920256   
947   ["'scalp defect']", "['congenital absence of s...             0.910375   
2165  ["'scalp defect']", "['congenital absence of s...             0.910375   
2304  ["'scalp defect']", "['congenital absence of s...             0.91


Saved per-sample metrics to: /content/drive/MyDrive/SmolVLM_runs/caption_vs_keywords_bertscore.csv


In [ ]:
import pandas as pd

# Paths
pred_path = f"{RUNS_DIR}/Smolvlm_Caption.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

# Load CSVs
pred = pd.read_csv(pred_path)
fitz = pd.read_csv(fitz_path)

# Normalize and extract md5 hash key
def extract_md5(x):
    if not isinstance(x, str):
        return ""
    x = x.split("/")[-1].split(".")[0]
    return x.lower()

pred["key"] = pred["ID"].apply(extract_md5)
fitz["key"] = fitz["md5hash"].str.lower()

# Merge — only keeps overlapping rows
merged = pred.merge(fitz, on="key", how="inner")
print(f"✅ Predicted captions: {len(pred)}")
print(f"✅ Fitzpatrick metadata: {len(fitz)}")
print(f"✅ Merged overlapping samples: {len(merged)}")

# Check tone distribution in your test subset
print("\nTone distribution in your test data:")
print(merged["fitzpatrick_scale"].value_counts().sort_index())


✅ Predicted captions: 3316
✅ Fitzpatrick metadata: 16577
✅ Merged overlapping samples: 3316

Tone distribution in your test data:
fitzpatrick_scale
-1    107
 1    588
 2    996
 3    634
 4    534
 5    322
 6    135
Name: count, dtype: int64


In [ ]:
def tone_group(tone):
    try:
        t = int(tone)
        if t in [1, 2]:
            return "Light"
        elif t in [3, 4]:
            return "Medium"
        elif t in [5, 6]:
            return "Dark"
        else:
            return "Unknown"
    except:
        return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)
print(merged["tone_group"].value_counts())


tone_group
Light      1584
Medium     1168
Dark        457
Unknown     107
Name: count, dtype: int64


BERTSCORE SKIN TONE AND KEYWORDS

In [ ]:
import pandas as pd

# === STEP 1: Paths ===
# Your per-sample metric file (from your screenshot)
bert_path = f"{RUNS_DIR}/caption_vs_keywords_bertscore.csv"

# Fitzpatrick metadata (contains md5hash + fitzpatrick_scale)
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

# === STEP 2: Load CSVs ===
bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# === STEP 3: Normalize keys for merge ===
def extract_md5(x):
    if not isinstance(x, str):
        return ""
    return x.split("/")[-1].split(".")[0].lower()

bert["key"] = bert["key"].astype(str).str.lower()  # already md5 in your file
fitz["key"] = fitz["md5hash"].str.lower()

# === STEP 4: Merge ===
merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# === STEP 5: Filter invalid tones (-1) ===
merged = merged[merged["fitzpatrick_scale"] > 0].copy()

# === STEP 6: Group into tone categories ===
def tone_group(t):
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    else:
        return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)
print(merged["tone_group"].value_counts())

# === STEP 7: Compute average fairness metrics ===
tone_summary = (
    merged.groupby("tone_group")[["bertscore_recall_kw","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone ===")
print(tone_summary)

# === STEP 8: Save merged CSV for visualization ===
out_path = f"{WORK_DIR}/bertscore_with_tone.csv"
merged.to_csv(out_path, index=False)
print(f"\nMerged dataset with tone info saved to:\n{out_path}")


Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316
tone_group
Light     1584
Medium    1168
Dark       457
Name: count, dtype: int64

=== Average Caption Quality by Skin Tone ===
  tone_group  bertscore_recall_kw  exact_keyword_coverage
0       Dark             0.803021                0.210642
1      Light             0.792575                0.165007
2     Medium             0.797725                0.197534

Merged dataset with tone info saved to:
/content/bertscore_with_tone.csv


In [ ]:
import pandas as pd

# === STEP 1: Paths ===
# Your per-sample metric file (from your screenshot)
bert_path = f"{RUNS_DIR}/caption_vs_keywords_bertscore.csv"

# Fitzpatrick metadata (contains md5hash + fitzpatrick_scale)
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

# === STEP 2: Load CSVs ===
bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# === STEP 3: Normalize keys for merge ===
def extract_md5(x):
    if not isinstance(x, str):
        return ""
    return x.split("/")[-1].split(".")[0].lower()

bert["key"] = bert["key"].astype(str).str.lower()  # already md5 in your file
fitz["key"] = fitz["md5hash"].str.lower()

# === STEP 4: Merge ===
merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# === STEP 5: Filter invalid tones (-1) ===
merged = merged[merged["fitzpatrick_scale"] > 0].copy()

# === STEP 6: Group into tone categories ===
def tone_group(t):
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    else:
        return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)
print(merged["tone_group"].value_counts())

# === STEP 7: Compute average fairness metrics ===
tone_summary = (
    merged.groupby("tone_group")[["bertscore_recall_kw","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone ===")
print(tone_summary)

# === STEP 8: Save merged CSV for visualization ===
out_path = f"{WORK_DIR}/bertscore_with_tone.csv"
merged.to_csv(out_path, index=False)
print(f"\nMerged dataset with tone info saved to:\n{out_path}")


BERTSCORE PER IMAGE CAPTION

In [ ]:
!pip install -q bert-score

import pandas as pd
import re, os, torch
from bert_score import score
from IPython.display import display

# ====== CSV paths ======
pred_path = f"{RUNS_DIR}/Smolvlm_Caption.csv"  # predicted captions
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"  # ground truth

# ====== Load CSVs ======
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

# Normalize column names
pred_df.columns = pred_df.columns.str.strip().str.lower()
gt_df.columns   = gt_df.columns.str.strip().str.lower()

# Rename columns for clarity
pred_df = pred_df.rename(columns={"id": "id_pred", "caption": "caption_pred"})
gt_df   = gt_df.rename(columns={"id": "id_true", "caption": "caption_true"})

# ====== Clean IDs (remove extensions) ======
def clean_id(x):
    x = str(x)
    x = os.path.basename(x)
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.lower().strip()

pred_df["key"] = pred_df["id_pred"].apply(clean_id)
gt_df["key"]   = gt_df["id_true"].apply(clean_id)

# ====== Merge on key ======
merged = pred_df.merge(gt_df[["key", "caption_true"]], on="key", how="inner")
print(f"Merged rows: {len(merged)}")
display(merged.head(5))

# ====== ImageCLEF-style caption preprocessing ======
def preprocess_caption(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

preds = [preprocess_caption(x) for x in merged["caption_pred"]]
refs  = [preprocess_caption(x) for x in merged["caption_true"]]

# ====== Compute BERTScore ======
P, R, F1 = score(
    preds, refs,
    model_type="roberta-large",
    lang="en",
    rescale_with_baseline=False,  # ImageCLEF-style raw recall
    idf=True,
    batch_size=16,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# ====== Summary metrics ======
print("\n=== ImageCLEF-style BERTScore ===")
print(f"Precision: {float(P.mean()):.4f}")
print(f"Recall:    {float(R.mean()):.4f}  <-- primary metric used by ImageCLEF")
print(f"F1:        {float(F1.mean()):.4f}")

# ====== Save per-sample scores ======
merged["bertscore_precision"] = [float(p) for p in P]
merged["bertscore_recall"]    = [float(r) for r in R]
merged["bertscore_f1"]        = [float(f) for f in F1]

# ====== Save merged + metrics ======
out_csv = f"{WORK_DIR}/merged_for_bertscore_with_metrics.csv"
merged.to_csv(out_csv, index=False)

print(f"\n✅ Per-sample BERTScore saved to:\n{out_csv}")
display(merged.head(10))


Merged rows: 3316


                                id_pred  \
0  0012821d6f11b96cf33f2c2ee5c68d1f.jpg   
1  001d22ff2543f95d2d38c18da0446c84.jpg   
2  002714e65a78f16fb05bc0aa95ea9761.jpg   
3  003e6abf20d234221a41b528b946e90c.jpg   
4  005b804472c7a27908f99e3d6d6cf91c.jpg   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a skin lesion with a reddish-b...   
2  The image shows a chronic inflammatory skin co...   
3  The image shows a person's arm with chronic it...   
4  The image shows a skin lesion with a blisters ...   

                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   
3  003e6abf20d234221a41b528b946e90c   
4  005b804472c7a27908f99e3d6d6cf91c   

                                        caption_true  
0  The image shows a close-up of a person's eye; ...  
1  The image shows a person's skin with a cluster...  
2  T

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== ImageCLEF-style BERTScore ===
Precision: 0.8417
Recall:    0.8417  <-- primary metric used by ImageCLEF
F1:        0.8415

✅ Per-sample BERTScore saved to:
/content/merged_for_bertscore_with_metrics.csv


                                id_pred  \
0  0012821d6f11b96cf33f2c2ee5c68d1f.jpg   
1  001d22ff2543f95d2d38c18da0446c84.jpg   
2  002714e65a78f16fb05bc0aa95ea9761.jpg   
3  003e6abf20d234221a41b528b946e90c.jpg   
4  005b804472c7a27908f99e3d6d6cf91c.jpg   
5  0060c63deaf2c1a022136b1f6b04b87d.jpg   
6  00918f1c84f591a61d5e59600db86f05.jpg   
7  0094c6373a5ac2add4f2b8bc78571243.jpg   
8  00a61ae0aa6d43a08152a7c4692ef9e2.jpg   
9  00c6529c53944cfc7b184abce798e16a.jpg   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a skin lesion with a reddish-b...   
2  The image shows a chronic inflammatory skin co...   
3  The image shows a person's arm with chronic it...   
4  The image shows a skin lesion with a blisters ...   
5  The image shows a close-up of a person's finge...   
6  The image depicts a cutaneous squamous cell ca...   
7  The image depicts a cutaneous squamous cell ca...   
8  The image shows a p

SKINTONE WITH BERTSCORE PREDICTION

In [ ]:
import pandas as pd

# === STEP 1: Paths ===
bertscore_path = f"{WORK_DIR}/merged_for_bertscore_with_metrics.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

# === STEP 2: Load CSVs ===
bertscore_df = pd.read_csv(bertscore_path)
fitz_df = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bertscore_df)} rows")
print(f"- Fitzpatrick metadata: {len(fitz_df)} rows")

# === STEP 3: Normalize merge keys ===
def clean_key(x):
    import os, re
    x = str(x)
    x = os.path.basename(x)
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.lower().strip()

bertscore_df["key"] = bertscore_df["key"].astype(str).str.lower().apply(clean_key)
fitz_df["key"] = fitz_df["md5hash"].astype(str).str.lower()

# === STEP 4: Merge datasets ===
merged = bertscore_df.merge(fitz_df, on="key", how="inner")
print(f"\nMerged dataset size: {len(merged)}")

# === STEP 5: Filter out invalid tones (-1) ===
merged = merged[merged["fitzpatrick_scale"] > 0].copy()

# === STEP 6: Define tone group ===
def tone_group(scale):
    scale = int(scale)
    if scale in [1, 2]:
        return "Light"
    elif scale in [3, 4]:
        return "Medium"
    elif scale in [5, 6]:
        return "Dark"
    else:
        return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)
print("\nTone distribution:")
print(merged["tone_group"].value_counts())

# === STEP 7: Compute average metrics ===
tone_summary = (
    merged.groupby("tone_group")[["bertscore_precision", "bertscore_recall", "bertscore_f1"]]
    .mean()
    .reset_index()
    .sort_values("tone_group")
)

print("\n=== Average BERTScore by Skin Tone ===")
print(tone_summary)

# === STEP 8: Save merged dataset with tone ===
out_csv = f"{WORK_DIR}/newbertscore_with_tone.csv"
merged.to_csv(out_csv, index=False)

print(f"\n✅ Merged dataset with tone info saved to:\n{out_csv}")


Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick metadata: 16577 rows

Merged dataset size: 3316

Tone distribution:
tone_group
Light     1584
Medium    1168
Dark       457
Name: count, dtype: int64

=== Average BERTScore by Skin Tone ===
  tone_group  bertscore_precision  bertscore_recall  bertscore_f1
0       Dark             0.845023          0.846498      0.845582
1      Light             0.840249          0.840011      0.839930
2     Medium             0.843074          0.842510      0.842619

✅ Merged dataset with tone info saved to:
/content/newbertscore_with_tone.csv


In [ ]:
!pip -q install "pandas==2.2.2" "numpy==2.1.3"
import os, sys
print("Restart runtime now: Runtime > Restart runtime")

Restart runtime now: Runtime > Restart runtime


In [ ]:
# =========================================
# 0) Paths
# =========================================
pred_path = f"{RUNS_DIR}/Smolvlm_Caption.csv"
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"

# =========================================
# 1) Install only what we need (NO -U)
# =========================================
!pip -q install evaluate bert-score rouge-score

import re, string
import numpy as np
import pandas as pd
from evaluate import load as hf_load
from bert_score import score as bertscore

# =========================================
# 2) Load CSVs
# =========================================
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

print("pred_df cols:", pred_df.columns.tolist())
print("gt_df cols  :", gt_df.columns.tolist())

# normalize column names
pred_df.columns = [c.strip().lower() for c in pred_df.columns]
gt_df.columns   = [c.strip().lower() for c in gt_df.columns]

# rename to standard
pred_df = pred_df.rename(columns={"id": "id", "caption": "pred_caption"})
gt_df   = gt_df.rename(columns={"id": "id", "caption": "gt_caption"})

assert {"id", "pred_caption"}.issubset(pred_df.columns), "Pred must have id & caption"
assert {"id", "gt_caption"}.issubset(gt_df.columns), "GT must have ID/Caption (any case)"

# =========================================
# 3) Normalize IDs (THIS usually fixes merge=0)
# =========================================
def normalize_id(x):
    x = str(x).strip()
    # remove common extensions if present
    x = re.sub(r"\.(jpg|jpeg|png|bmp|tif|tiff)$", "", x, flags=re.IGNORECASE)
    return x

pred_df["id_norm"] = pred_df["id"].map(normalize_id)
gt_df["id_norm"]   = gt_df["id"].map(normalize_id)

# Diagnostics: check overlap
pred_ids = set(pred_df["id_norm"].unique())
gt_ids   = set(gt_df["id_norm"].unique())
overlap  = pred_ids.intersection(gt_ids)

print("\nUnique pred ids:", len(pred_ids))
print("Unique gt ids  :", len(gt_ids))
print("Overlap ids    :", len(overlap))

print("\nExample pred ids:", list(pred_ids)[:5])
print("Example gt ids  :", list(gt_ids)[:5])

# show some ids that don't match
print("\nSome pred-only ids:", list(pred_ids - gt_ids)[:10])
print("Some gt-only ids  :", list(gt_ids - pred_ids)[:10])

# =========================================
# 4) Merge using normalized ids
# =========================================
df = pd.merge(
    gt_df[["id_norm", "gt_caption"]],
    pred_df[["id_norm", "pred_caption"]],
    on="id_norm",
    how="inner"
).dropna(subset=["gt_caption", "pred_caption"]).reset_index(drop=True)

print("\nMerged rows:", len(df))
if len(df) == 0:
    raise ValueError("Merged rows is 0. Your IDs still don't match. Check the printed diagnostics above.")

# =========================================
# 5) CLEF-style preprocessing
# =========================================
punct_table = str.maketrans("", "", string.punctuation)

def clef_preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"\d+(\.\d+)?", "number", text)
    text = text.translate(punct_table)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["gt_pp"]   = df["gt_caption"].map(clef_preprocess)
df["pred_pp"] = df["pred_caption"].map(clef_preprocess)

# =========================================
# 6) ROUGE-1 (F1)
# =========================================
rouge = hf_load("rouge")
rouge_res = rouge.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist(),
    rouge_types=["rouge1"]
)
rouge1_f1 = float(rouge_res["rouge1"])

# =========================================
# 7) BERTScore (Recall, IDF)
# =========================================
P, R, F = bertscore(
    cands=df["pred_pp"].tolist(),
    refs=df["gt_pp"].tolist(),
    model_type="microsoft/deberta-xlarge-mnli",
    lang="en",
    idf=True,
    batch_size=16,
    verbose=True
)
bertscore_recall = float(R.mean().item())

# =========================================
# 8) Print results
# =========================================
print("\n================ RESULTS ================\n")
print(f"ROUGE-1 (F1):              {rouge1_f1:.6f}")
print(f"BERTScore (Recall, IDF):   {bertscore_recall:.6f}")

# =========================================
# 9) Save merged + preprocessed file
# =========================================
out_path = f"{WORK_DIR}/clef_eval_merged.csv"
df.to_csv(out_path, index=False)
print("\nSaved merged eval file:", out_path)


pred_df cols: ['ID', 'Caption']
gt_df cols  : ['ID', 'Caption']

Unique pred ids: 3316
Unique gt ids  : 3316
Overlap ids    : 3316

Example pred ids: ['396357938addc364fe5d70a5d4d92a36', '60670f2264dc573297062442b00f4ebf', 'be506d4a799385099ac85daa695d6df3', 'fecff967696d462b75d70a794fdfd9e5', '55120bb72ff415aef6ffa03bcc3d88e1']
Example gt ids  : ['be506d4a799385099ac85daa695d6df3', '60670f2264dc573297062442b00f4ebf', '396357938addc364fe5d70a5d4d92a36', 'fecff967696d462b75d70a794fdfd9e5', '55120bb72ff415aef6ffa03bcc3d88e1']

Some pred-only ids: []
Some gt-only ids  : []

Merged rows: 3316


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

preparing IDF dict...
done in 2.46 seconds
calculating scores...
computing bert embedding.


  0%|          | 0/318 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/208 [00:00<?, ?it/s]

done in 50.46 seconds, 65.72 sentences/sec

================ RESULTS ================

ROUGE-1 (F1):              0.496235
BERTScore (Recall, IDF):   0.620543

Saved merged eval file: /content/clef_eval_merged.csv


In [ ]:
!pip -q install git+https://github.com/google-research/bleurt.git


  Preparing metadata (setup.py) ... done


In [ ]:
!pip -q install git+https://github.com/google-research/bleurt.git

  Preparing metadata (setup.py) ... done


In [ ]:
import pandas as pd
import numpy as np
from evaluate import load as hf_load

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")  # your merged file

bleurt = hf_load("bleurt", checkpoint="BLEURT-20")

bleurt_scores = bleurt.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist()
)["scores"]

df["bleurt20"] = bleurt_scores
print("BLEURT-20 (avg):", round(float(np.mean(bleurt_scores)), 6))

df.to_csv(f"{WORK_DIR}/clef_eval_merged_plus_bleurt.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_merged_plus_bleurt.csv")


BLEURT-20 (avg): -0.384961
Saved: /content/clef_eval_merged_plus_bleurt.csv


In [ ]:
# ===============================
# BLEURT-20 (NO MINUS reporting)
# ===============================

!pip -q install git+https://github.com/google-research/bleurt.git
!pip -q install -q evaluate

import pandas as pd
import numpy as np
from evaluate import load as hf_load

# 1) Load merged file you already saved
df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")

# safety
assert "pred_pp" in df.columns and "gt_pp" in df.columns, "Missing pred_pp / gt_pp in clef_eval_merged.csv"

# 2) BLEURT-20 raw (can be negative, that's normal)
bleurt = hf_load("bleurt", checkpoint="BLEURT-20")
bleurt_raw = bleurt.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist()
)["scores"]

df["bleurt20_raw"] = bleurt_raw

raw_mean = float(np.mean(bleurt_raw))
raw_min  = float(np.min(bleurt_raw))
raw_max  = float(np.max(bleurt_raw))

print("BLEURT-20 raw mean:", round(raw_mean, 6))
print("BLEURT-20 raw min :", round(raw_min, 6))
print("BLEURT-20 raw max :", round(raw_max, 6))

# 3) NO-MINUS version A: shifted to be >= 0
#    (smallest becomes 0)
df["bleurt20_shifted"] = df["bleurt20_raw"] - raw_min
shifted_mean = float(df["bleurt20_shifted"].mean())

print("\nBLEURT-20 shifted mean (>=0):", round(shifted_mean, 6))
print("Shifted min:", round(float(df['bleurt20_shifted'].min()), 6))

# 4) NO-MINUS version B (recommended): normalize to [0, 1]
#    (best for averaging with ROUGE/BERTScore)
den = (raw_max - raw_min) if (raw_max - raw_min) != 0 else 1e-12
df["bleurt20_norm01"] = (df["bleurt20_raw"] - raw_min) / den
norm_mean = float(df["bleurt20_norm01"].mean())

print("\nBLEURT-20 normalized [0,1] mean:", round(norm_mean, 6))
print("Norm min:", round(float(df['bleurt20_norm01'].min()), 6),
      "Norm max:", round(float(df['bleurt20_norm01'].max()), 6))

# 5) Save
out_path = f"{WORK_DIR}/clef_eval_merged_plus_bleurt_nominas.csv"
df.to_csv(out_path, index=False)
print("\nSaved:", out_path)

# If you want a single BLEURT number with no minus for reporting, use:
print("\nREPORT THIS (no minus): BLEURT-20_norm01 =", round(norm_mean, 6))



  Preparing metadata (setup.py) ... done


BLEURT-20 raw mean: -0.384961
BLEURT-20 raw min : -1.408898
BLEURT-20 raw max : 0.935596

BLEURT-20 shifted mean (>=0): 1.023937
Shifted min: 0.0

BLEURT-20 normalized [0,1] mean: 0.436741
Norm min: 0.0 Norm max: 1.0

Saved: /content/clef_eval_merged_plus_bleurt_nominas.csv

REPORT THIS (no minus): BLEURT-20_norm01 = 0.436741


In [ ]:
!pip -q install transformers accelerate --no-deps

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged_plus_bleurt_nominas.csv")

premises   = df["gt_pp"].astype(str).tolist()     # reference/context
hypotheses = df["pred_pp"].astype(str).tolist()   # claim/prediction

model_name = "microsoft/deberta-large-mnli"  # ✅ correct model (no 404)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

id2label = {int(k): v for k, v in model.config.id2label.items()}
print("id2label:", id2label)

# find entailment index robustly
entail_idx = None
for k, v in id2label.items():
    if str(v).lower().startswith("entail"):
        entail_idx = k
        break
if entail_idx is None:
    entail_idx = 2  # common MNLI ordering

def batch_entailment(premises, hypotheses, batch_size=16, max_len=256):
    scores = []
    for i in range(0, len(premises), batch_size):
        p = premises[i:i+batch_size]
        h = hypotheses[i:i+batch_size]
        enc = tokenizer(
            p, h, truncation=True, padding=True, max_length=max_len,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        scores.extend(probs[:, entail_idx].tolist())
    return np.array(scores)

ent_scores = batch_entailment(premises, hypotheses, batch_size=16, max_len=256)
df["nli_align_entail"] = ent_scores

print("NLI-Align (Entailment prob) avg:", round(float(ent_scores.mean()), 6))
print("min:", round(float(ent_scores.min()), 6), "max:", round(float(ent_scores.max()), 6))

out_path = f"{WORK_DIR}/clef_eval_with_nli_align.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


id2label: {0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}
NLI-Align (Entailment prob) avg: 0.137283
min: 0.000101 max: 0.996377
Saved: /content/clef_eval_with_nli_align.csv


In [ ]:
import pandas as pd, numpy as np

rouge1_f1 = 0.496125
bertscore_recall =  0.620543
df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_nli_align.csv")

bleurt_norm = float(df["bleurt20_norm01"].mean())   # 0..1
nli_align   = float(df["nli_align_entail"].mean())  # 0..1

final_avg_4 = float(np.mean([rouge1_f1, bertscore_recall, bleurt_norm, nli_align]))

print("\n===== FINAL SCORE (4 metrics, no minus) =====")
print(f"ROUGE-1 (F1):               {rouge1_f1:.6f}")
print(f"BERTScore (Recall, IDF):    {bertscore_recall:.6f}")
print(f"BLEURT-20_norm01 (avg):     {bleurt_norm:.6f}")
print(f"NLI-Align Entail (avg):     {nli_align:.6f}")
print(f"\nAverage over 4 metrics:     {final_avg_4:.6f}")



===== FINAL SCORE (4 metrics, no minus) =====
ROUGE-1 (F1):               0.496125
BERTScore (Recall, IDF):    0.620543
BLEURT-20_norm01 (avg):     0.436741
NLI-Align Entail (avg):     0.137283

Average over 4 metrics:     0.422673


In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter

# ---------------------------
# 0) Load your existing eval file
# ---------------------------
df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_nli_align.csv")

# You already have these two as constants from your earlier run:
rouge1_f1 = 0.496235
bertscore_recall_idf = 0.620543

# BLEURT normalized (0..1, no minus) should already exist from your BLEURT cell
assert "bleurt20_norm01" in df.columns, "Missing bleurt20_norm01. Run BLEURT no-minus code first."
bleurt_norm = float(df["bleurt20_norm01"].mean())

# NLI align entailment (0..1)
assert "nli_align_entail" in df.columns, "Missing nli_align_entail. Run NLI-align code first."
nli_align = float(df["nli_align_entail"].mean())

print("Loaded rows:", len(df))





# ---------------------------
# 2) Image–Caption Similarity (optional)
#    - Requires an image path column in df
#    - If you have images, set IMAGE_COL to your column name.
# ---------------------------
IMAGE_COL = None  # e.g., "image_path"  (set this if you have it)

sim_avg = None
if IMAGE_COL is not None and IMAGE_COL in df.columns:
    !pip -q install open_clip_torch pillow

    import torch
    import open_clip
    from PIL import Image

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # A practical CLIP baseline (not medical-specific). If you want BioMedCLIP later, tell me.
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    model = model.to(device).eval()

    def clip_similarity(image_path, caption):
        try:
            img = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
        except:
            return np.nan
        text = tokenizer([caption]).to(device)
        with torch.no_grad():
            img_feat = model.encode_image(img)
            txt_feat = model.encode_text(text)
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
            txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
            sim = (img_feat @ txt_feat.T).squeeze().item()
        return sim

    sims = []
    for p, cap in zip(df[IMAGE_COL].astype(str), df["pred_pp"].astype(str)):
        sims.append(clip_similarity(p, cap))

    df["img_caption_sim"] = sims
    sim_avg = float(np.nanmean(sims))
else:
    print("\n[INFO] Image–Caption similarity skipped (no IMAGE_COL set / not present).")


# ---------------------------
# 3) Compute the requested averages
# ---------------------------

# Relevance metrics: ROUGE, BERTScore, BLEURT_norm, Similarity
# If similarity wasn't computed, we compute relevance over the available 3.
relevance_parts = [
    ("ROUGE-1_F1", rouge1_f1),
    ("BERTScore_Recall_IDF", bertscore_recall_idf),
    ("BLEURT20_norm01", bleurt_norm),
]

if sim_avg is not None:
    relevance_parts.append(("ImgCaptionSimilarity", sim_avg))

relevance_avg = float(np.mean([v for _, v in relevance_parts]))

# Factuality metrics: UMLS_F1 + NLI-align


# Overall: average of relevance_avg and factuality_avg (clean 2-aspect CLEF-style)
overall_score = float(np.mean([relevance_avg, factuality_avg]))

print("\n==================== FINAL REPORT ====================\n")

print("Relevance metrics:")
for k, v in relevance_parts:
    print(f"  {k}: {v:.6f}")
print(f"  Relevance average: {relevance_avg:.6f}")

print("\nFactuality metrics:")
#print(f"  UMLS Concept F1 (avg): {umls_f1_avg:.6f}   [{umls_mode}]")
print(f"  NLI-Align entail (avg): {nli_align:.6f}")

print("\nOverall:")
print(f"  Overall score (avg of relevance & factuality): {overall_score:.6f}")

# Save a final file
out_path = f"{WORK_DIR}/clef_eval_final_report.csv"
df.to_csv(out_path, index=False)
print("\nSaved per-sample file:", out_path)


Loaded rows: 3316

[INFO] Image–Caption similarity skipped (no IMAGE_COL set / not present).

==================== FINAL REPORT ====================

Relevance metrics:
  ROUGE-1_F1: 0.496235
  BERTScore_Recall_IDF: 0.620543
  BLEURT20_norm01: 0.436741
  Relevance average: 0.517840

Factuality metrics:
  NLI-Align entail (avg): 0.137283

Overall:
  Overall score (avg of relevance & factuality): 0.395122

Saved per-sample file: /content/clef_eval_final_report.csv


In [ ]:
!pip -q install "transformers==4.44.2" --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 100.0 MB/s eta 0:00:00


In [ ]:
import os, glob
import pandas as pd

pred_path = f"{RUNS_DIR}/Smolvlm_Caption.csv"
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"

pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

print("pred_df cols:", pred_df.columns.tolist())
print("gt_df cols  :", gt_df.columns.tolist())


pred_df cols: ['ID', 'Caption']
gt_df cols  : ['ID', 'Caption']


In [ ]:
pred_df = pred_df.rename(columns={"ID": "id", "caption": "caption_pred"})
gt_df   = gt_df.rename(columns={"ID": "id", "Caption": "caption_true"})

pred_df["id"] = pred_df["id"].astype(str).str.strip().str.replace(".jpg", "", regex=False)
gt_df["id"]   = gt_df["id"].astype(str).str.strip().str.replace(".jpg", "", regex=False)

df = gt_df.merge(pred_df, on="id", how="inner")
print("Merged rows:", len(df))
df.head()

Merged rows: 3316


                                 id  \
0  d395430d11d4ac72e6f60360aabf0e61   
1  8dffbb994ef17963995f8059a76418b9   
2  01be7f7454385c1abaa9d10aabcaa751   
3  e1e0b7f3462e4d9c5819e81c22d8238d   
4  c7fcb5f49fbe7fb7eeec7eaf196b299a   

                                        caption_true  \
0  The image shows a person with generalized pust...   
1  The lesion is located on the skin;  possibly o...   
2  The image shows a close-up of a foot;  specifi...   
3  The image shows a skin lesion with a reddish-b...   
4  The image shows a person with a rash that appe...   

                                             Caption  
0  The image shows a person with chronic itchy sk...  
1  The image depicts a cutaneous squamous cell ca...  
2  The image shows a close-up of a person's skin;...  
3  The visible findings include confluent and ret...  
4  The image shows a skin lesion with a focal;  i...  

In [ ]:
df.to_csv(f"{WORK_DIR}/clef_eval_merged.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_merged.csv")


Saved: /content/clef_eval_merged.csv


In [ ]:
import os
import glob
import pandas as pd

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")

IMG_ROOT = f"{CAPTIONS_DIR}/Test-image"

# ✅ safer glob (directly search jpg files)
all_imgs = glob.glob(os.path.join(IMG_ROOT, "*.jpg"))
print("Found JPG images:", len(all_imgs))

# Map filename -> path
img_map = {os.path.splitext(os.path.basename(p))[0]: p for p in all_imgs}

df["image_path"] = df["id"].astype(str).map(img_map)

matched = df["image_path"].notna().sum()
print("Matched images:", matched, "/", len(df))

df.to_csv(f"{WORK_DIR}/clef_eval_merged_with_paths.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_merged_with_paths.csv")


Found JPG images: 3316
Matched images: 3316 / 3316
Saved: /content/clef_eval_merged_with_paths.csv


In [ ]:
!pip -q install open_clip_torch pillow --no-deps


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.8 MB/s eta 0:00:00


In [ ]:
!pip -q install ftfy regex


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
import open_clip

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged_with_paths.csv")
df_ok = df.dropna(subset=["image_path"]).copy()

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
model, _, preprocess = open_clip.create_model_and_transforms(model_id)
tokenizer = open_clip.get_tokenizer(model_id)
model = model.to(device).eval()

def sim_one(img_path, caption):
    img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    txt = tokenizer([str(caption)]).to(device)

    with torch.no_grad():
        imf = model.encode_image(img)
        txf = model.encode_text(txt)

        imf = imf / imf.norm(dim=-1, keepdim=True)
        txf = txf / txf.norm(dim=-1, keepdim=True)

        return float((imf @ txf.T).squeeze().item())

# use predicted caption text for similarity
sims = []
for p, cap in zip(df_ok["image_path"], df_ok["Caption"]):
    sims.append(sim_one(p, cap))

df_ok["img_caption_sim"] = sims
sim_avg = float(np.mean(sims))

print("✅ Image–Caption Similarity avg (BiomedCLIP):", round(sim_avg, 6))

df.loc[df_ok.index, "img_caption_sim"] = df_ok["img_caption_sim"].values
df.to_csv(f"{WORK_DIR}/clef_eval_with_similarity.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_with_similarity.csv")


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ Image–Caption Similarity avg (BiomedCLIP): 0.374994
Saved: /content/clef_eval_with_similarity.csv


In [ ]:
import pandas as pd
import numpy as np

# file that has similarity
sim_df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_similarity.csv")

# file that has bleurt + nli (use your final report)
base_df = pd.read_csv(f"{WORK_DIR}/clef_eval_final_report.csv")

# ---- detect ID column in each file ----
def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

id_sim  = pick_first(sim_df,  ["id", "id_norm", "id_true"])
id_base = pick_first(base_df, ["id", "id_norm", "id_true"])

print("ID cols:", id_sim, id_base)

# normalize IDs for safe merge
sim_df["id_key"]  = sim_df[id_sim].astype(str).str.strip().str.replace(".jpg","",regex=False)
base_df["id_key"] = base_df[id_base].astype(str).str.strip().str.replace(".jpg","",regex=False)

# ---- merge similarity into base ----
df = base_df.merge(
    sim_df[["id_key", "img_caption_sim"]],
    on="id_key",
    how="left"
)

# fixed values you already computed
ROUGE1_F1 = 0.496125
BERTSCORE_RECALL = 0.620543

# columns in base_df
BLEURT_COL = pick_first(df, ["bleurt20_norm01", "BLEURT20_norm01"])
ALIGN_COL  = pick_first(df, ["nli_align_entail", "NLI_align_entail"])

BLEURT = float(df[BLEURT_COL].mean())
ALIGNSCORE = float(df[ALIGN_COL].mean())
SIM = float(df["img_caption_sim"].dropna().mean())  # now available after merge

# CLEF relevance avg includes similarity
RELEVANCE_AVG = float(np.mean([ROUGE1_F1, BERTSCORE_RECALL, BLEURT, SIM]))

# No UMLS → factuality = AlignScore only
FACTUALITY_AVG = float(ALIGNSCORE)

# Overall = avg(relevance_avg, factuality_avg)
OVERALL = float(np.mean([RELEVANCE_AVG, FACTUALITY_AVG]))

row = {
    "ID": 1,
    "Submission Name": "smolvlm",
    "Overall": round(OVERALL, 6),
    "Similarity": round(SIM, 6),
    "BERTScore (Recall)": round(BERTSCORE_RECALL, 6),
    "ROUGE-1": round(ROUGE1_F1, 6),
    "BLEURT": round(BLEURT, 6),
    "Relevance Average": round(RELEVANCE_AVG, 6),
    "UMLS Concept F1": "-",     # removed
    "AlignScore": round(ALIGNSCORE, 6),
    "Factuality Average": round(FACTUALITY_AVG, 6),
}

leaderboard = pd.DataFrame([row])
leaderboard


ID cols: id id_norm


   ID Submission Name   Overall  Similarity  BERTScore (Recall)  \
0   1 Smolvlm  0.309692    0.374994            0.620543   

    ROUGE-1    BLEURT  Relevance Average UMLS Concept F1  AlignScore  \
0  0.496125  0.436741           0.482101               -    0.137283   

   Factuality Average  
0            0.137283  

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

# ---------------------------
# 1) FILES
# ---------------------------
train_kw_path = f"{WORK_DIR}/train.csv"   # 11k+
test_eval_path = f"{WORK_DIR}/clef_eval_merged.csv"                 # has ID + caption_true/pred
test_label_path = f"{WORK_DIR}/test.csv"   # has ID + label_name (common label)

# ---------------------------
# 2) COLUMNS (CHANGE THESE)
# ---------------------------
TRAIN_ID_COL = "image_id"
TRAIN_LABEL_COL = "label"     # common label name
TRAIN_KW_COL = "concepts"          # could be "keyword" or list-like string

TEST_ID_COL = "md5hash"
TEST_LABEL_COL = "label"

GT_CAP_COL = "caption_true"
PRED_CAP_COL = "caption"

# ---------------------------
# 3) Helpers
# ---------------------------
def norm_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_keywords(x):
    """Handles: 'k1;k2', 'k1, k2', "['k1','k2']", etc."""
    if pd.isna(x): return []
    s = str(x).strip()
    # list-like
    if s.startswith("[") and s.endswith("]"):
        s = s.strip("[]")
    # split by common separators
    parts = re.split(r"[;,|]\s*|,\s*", s)
    parts = [p.strip().lower() for p in parts if p.strip()]
    return parts

def build_patterns(vocab):
    vocab = sorted(set([v.strip().lower() for v in vocab if len(str(v).strip()) >= 2]),
                   key=len, reverse=True)
    return [(k, re.compile(rf"(?<!\w){re.escape(k)}(?!\w)")) for k in vocab]

def extract_from_vocab(text, patterns):
    t = norm_text(text)
    found = set()
    for k, pat in patterns:
        if pat.search(t):
            found.add(k)
    return found

def f1_from_sets(pred_set, gt_set):
    if len(pred_set)==0 and len(gt_set)==0:
        return 1.0
    if len(pred_set)==0 or len(gt_set)==0:
        return 0.0
    tp = len(pred_set & gt_set)
    p = tp/len(pred_set) if pred_set else 0.0
    r = tp/len(gt_set) if gt_set else 0.0
    return (2*p*r/(p+r)) if (p+r) else 0.0

# ---------------------------
# 4) Load train keywords and build label->keywords map
# ---------------------------
train_df = pd.read_csv(train_kw_path)
train_df.columns = train_df.columns.str.lower().str.strip()

# if your real columns are different, update the names above
train_df = train_df.rename(columns={
    TRAIN_ID_COL.lower(): "image_id",
    TRAIN_LABEL_COL.lower(): "label",
    TRAIN_KW_COL.lower(): "concepts"
})

label_kw_counter = defaultdict(Counter)

for _, row in train_df.iterrows():
    lbl = str(row["label"]).strip().lower()
    kws = split_keywords(row["concepts"])
    for k in kws:
        label_kw_counter[lbl][k] += 1

# keep top-K keywords per label (tune K)
TOPK = 30
label_kw_map = {lbl: set([k for k,_ in cnt.most_common(TOPK)]) for lbl, cnt in label_kw_counter.items()}

print("Labels in train:", len(label_kw_map))
print("Example label keywords:", list(label_kw_map.items())[:1])

# optional: global vocab from all label keywords
global_vocab = set().union(*label_kw_map.values())
patterns = build_patterns(global_vocab)
print("Global keyword vocab size:", len(global_vocab))

# ---------------------------
# 5) Load test labels + eval captions, align by ID
# ---------------------------
test_labels = pd.read_csv(test_label_path)
test_labels.columns = test_labels.columns.str.lower().str.strip()
test_labels = test_labels.rename(columns={TEST_ID_COL.lower():"id", TEST_LABEL_COL.lower():"label_name"})
test_labels["id"] = test_labels["id"].astype(str).str.strip()

eval_df = pd.read_csv(test_eval_path)
eval_df.columns = eval_df.columns.str.lower().str.strip()

# adjust if needed
if "id" not in eval_df.columns and "id_norm" in eval_df.columns:
    eval_df["id"] = eval_df["id_norm"]

eval_df["id"] = eval_df["id"].astype(str).str.strip()

df = eval_df.merge(test_labels[["id","label_name"]], on="id", how="left")
df["label_name"] = df["label_name"].astype(str).str.strip().str.lower()

# ---------------------------
# 6) Metric A: Keyword F1 between GT vs Pred captions (global keyword vocab)
# ---------------------------
gt_sets = df[GT_CAP_COL.lower()].fillna("").astype(str).apply(lambda x: extract_from_vocab(x, patterns))
pr_sets = df[PRED_CAP_COL.lower()].fillna("").astype(str).apply(lambda x: extract_from_vocab(x, patterns))

df["kw_f1_gt_vs_pred"] = [f1_from_sets(p,g) for p,g in zip(pr_sets, gt_sets)]
print("Keyword F1 (GT vs Pred) avg:", round(float(df["kw_f1_gt_vs_pred"].mean()), 6))

# ---------------------------
# 7) Metric B (optional but useful): Label-grounding score
#     Pred keywords vs label-derived keyword set
# ---------------------------
label_sets = df["label_name"].apply(lambda l: label_kw_map.get(l, set()))
df["kw_f1_pred_vs_label"] = [f1_from_sets(p, lab) for p,lab in zip(pr_sets, label_sets)]
print("Keyword F1 (Pred vs Label) avg:", round(float(df["kw_f1_pred_vs_label"].mean()), 6))

df.to_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics.csv", index=False)
print("Saved: /content/clef_eval_with_label_keyword_metrics.csv")


Labels in train: 114
Example label keywords: [('hidradenitis', {"'boil-like nodules.'", "'purulent discharge'", "'scarring'", "'persistent nodules'", "'sinuses'", "'abscesses'", "'recurrent nodules'", "'chronic inflammatory skin condition'"})]
Global keyword vocab size: 791
Keyword F1 (GT vs Pred) avg: 1.0
Keyword F1 (Pred vs Label) avg: 0.0
Saved: /content/clef_eval_with_label_keyword_metrics.csv


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics.csv")

train_kw_path = f"{WORK_DIR}/train.csv"  # <-- put your real file
train_df = pd.read_csv(train_kw_path)
train_df.columns = train_df.columns.str.lower().str.strip()

# change these if needed
train_df = train_df.rename(columns={
    "label": "label_name",
    "concepts": "keywords"
})

def norm_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_keywords(x):
    if pd.isna(x): return []
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s.strip("[]")
    parts = re.split(r"[;,|]\s*|,\s*", s)
    parts = [p.strip() for p in parts if p.strip()]
    return parts

# build label->keyword counter (BUT normalize each keyword!)
label_kw_counter = defaultdict(Counter)

for _, row in train_df.iterrows():
    lbl = norm_text(row["label_name"])
    kws = split_keywords(row["keywords"])
    for k in kws:
        k2 = norm_text(k)      # ✅ normalize keyword
        if len(k2) >= 2:
            label_kw_counter[lbl][k2] += 1

TOPK = 30
label_kw_map = {lbl: set([k for k,_ in cnt.most_common(TOPK)]) for lbl, cnt in label_kw_counter.items()}

global_vocab = set().union(*label_kw_map.values())

print("Labels:", len(label_kw_map))
print("Global vocab:", len(global_vocab))
print("Example normalized keywords:", list(label_kw_map.items())[:1])

# regex patterns from normalized keywords
vocab_sorted = sorted(global_vocab, key=len, reverse=True)
patterns = [(k, re.compile(rf"(?<!\w){re.escape(k)}(?!\w)")) for k in vocab_sorted]

def extract_vocab(text):
    t = norm_text(text)
    found = set()
    for k, pat in patterns:
        if pat.search(t):
            found.add(k)
    return found

def f1_from_sets(pred_set, gt_set):
    if len(pred_set)==0 and len(gt_set)==0:
        return np.nan  # ✅ IMPORTANT: do NOT give 1.0 for empty-empty
    if len(pred_set)==0 or len(gt_set)==0:
        return 0.0
    tp = len(pred_set & gt_set)
    p = tp/len(pred_set)
    r = tp/len(gt_set)
    return (2*p*r/(p+r)) if (p+r) else 0.0

# detect columns
gt_col = "caption_true" if "caption_true" in df.columns else "caption_gt"
pred_col = "caption" if "caption" in df.columns else "caption"

df["kw_set_gt"] = df[gt_col].fillna("").astype(str).apply(extract_vocab)
df["kw_set_pred"] = df[pred_col].fillna("").astype(str).apply(extract_vocab)

df["kw_f1_gt_vs_pred"] = [
    f1_from_sets(p,g) for p,g in zip(df["kw_set_pred"], df["kw_set_gt"])
]

print("\nKeyword F1 (GT vs Pred) avg (ignoring NaN):",
      round(float(np.nanmean(df["kw_f1_gt_vs_pred"])), 6))

# Show how many are empty sets
print("Empty GT keyword sets:", (df["kw_set_gt"].apply(len)==0).sum(), "/", len(df))
print("Empty Pred keyword sets:", (df["kw_set_pred"].apply(len)==0).sum(), "/", len(df))

df.to_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics_FIXED.csv", index=False)
print("Saved: /content/clef_eval_with_label_keyword_metrics_FIXED.csv")


Labels: 114
Global vocab: 784
Example normalized keywords: [('hidradenitis', {'scarring', 'recurrent nodules', 'chronic inflammatory skin condition', 'sinuses', 'boil like nodules', 'abscesses', 'persistent nodules', 'purulent discharge'})]

Keyword F1 (GT vs Pred) avg (ignoring NaN): 0.345923
Empty GT keyword sets: 2 / 3316
Empty Pred keyword sets: 0 / 3316
Saved: /content/clef_eval_with_label_keyword_metrics_FIXED.csv


In [ ]:
import pandas as pd
import numpy as np

# =============================
# Files
# =============================
sim_df  = pd.read_csv(f"{WORK_DIR}/clef_eval_with_similarity.csv")                 # has similarity per id
base_df = pd.read_csv(f"{WORK_DIR}/clef_eval_final_report.csv")                   # has bleurt + nli_align_entail
kw_df   = pd.read_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics_FIXED.csv")  # has kw_f1_gt_vs_pred

# =============================
# Helper: pick a column safely
# =============================
def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# normalize column names (important!)
sim_df.columns  = sim_df.columns.str.strip()
base_df.columns = base_df.columns.str.strip()
kw_df.columns   = kw_df.columns.str.strip()

# =============================
# Detect ID columns
# =============================
id_sim  = pick_first(sim_df,  ["id", "id_norm", "id_true", "ID"])
id_base = pick_first(base_df, ["id", "id_norm", "id_true", "ID"])
id_kw   = pick_first(kw_df,   ["id", "id_norm", "id_true", "ID"])

print("ID cols:", id_sim, id_base, id_kw)

# =============================
# Create merge keys
# =============================
def norm_id(x):
    x = str(x).strip()
    return x.replace(".jpg","").replace(".png","")

sim_df["id_key"]  = sim_df[id_sim].apply(norm_id)
base_df["id_key"] = base_df[id_base].apply(norm_id)
kw_df["id_key"]   = kw_df[id_kw].apply(norm_id)

# =============================
# Detect similarity column
# =============================
SIM_COL = pick_first(sim_df, ["img_caption_sim", "similarity", "sim"])
print("SIM_COL:", SIM_COL)

# merge similarity into base
df = base_df.merge(sim_df[["id_key", SIM_COL]], on="id_key", how="left")

# merge keyword f1 too
KWF1_COL = pick_first(kw_df, ["kw_f1_gt_vs_pred", "kw_f1"])
print("KWF1_COL:", KWF1_COL)

df = df.merge(kw_df[["id_key", KWF1_COL]], on="id_key", how="left")

# =============================
# Use your already computed constants
# =============================
ROUGE1_F1 = 0.496125
BERTSCORE_RECALL = 0.620543

# detect bleurt + align columns in base
BLEURT_COL = pick_first(df, ["bleurt20_norm01", "BLEURT20_norm01", "bleurt"])
ALIGN_COL  = pick_first(df, ["nli_align_entail", "NLI_align_entail", "alignscore", "AlignScore"])

print("BLEURT_COL:", BLEURT_COL)
print("ALIGN_COL :", ALIGN_COL)

BLEURT = float(pd.to_numeric(df[BLEURT_COL], errors="coerce").mean())
ALIGNSCORE = float(pd.to_numeric(df[ALIGN_COL], errors="coerce").mean())
SIM = float(pd.to_numeric(df[SIM_COL], errors="coerce").mean())

# keyword f1 (ignore NaNs)
DERM_KWF1 = float(np.nanmean(pd.to_numeric(df[KWF1_COL], errors="coerce")))

# =============================
# CLEF-like averages
# =============================
RELEVANCE_AVG = float(np.mean([ROUGE1_F1, BERTSCORE_RECALL, BLEURT, SIM]))

# factuality avg WITHOUT UMLS:
# Option 1: Only AlignScore (CLEF-like fallback)
# FACTUALITY_AVG = ALIGNSCORE

# Option 2 (recommended): AlignScore + Derm Keyword Concept F1
FACTUALITY_AVG = float(np.mean([ALIGNSCORE, DERM_KWF1]))

OVERALL = float(np.mean([RELEVANCE_AVG, FACTUALITY_AVG]))

# =============================
# Print CLEF-style row
# =============================
row = {
    "ID": 1,
    "Submission Name": "smolvlm",
    "Overall": round(OVERALL, 6),
    "Similarity": round(SIM, 6),
    "BERTScore (Recall)": round(BERTSCORE_RECALL, 6),
    "ROUGE-1": round(ROUGE1_F1, 6),
    "BLEURT": round(BLEURT, 6),
    "Relevance Average": round(RELEVANCE_AVG, 6),
    "Derm Keyword Concept F1": round(DERM_KWF1, 6),
    "AlignScore": round(ALIGNSCORE, 6),
    "Factuality Average": round(FACTUALITY_AVG, 6),
}

leaderboard = pd.DataFrame([row])
leaderboard




ID cols: id id_norm id
SIM_COL: img_caption_sim
KWF1_COL: kw_f1_gt_vs_pred
BLEURT_COL: bleurt20_norm01
ALIGN_COL : nli_align_entail


   ID Submission Name   Overall  Similarity  BERTScore (Recall)  \
0   1 smolvlm  0.361852    0.374994            0.620543   

    ROUGE-1    BLEURT  Relevance Average  Derm Keyword Concept F1  AlignScore  \
0  0.496125  0.436741           0.482101                 0.345923    0.137283   

   Factuality Average  
0            0.241603  

In [ ]:
!pip -q install -U bert-score


In [ ]:
# =========================================
# CLEF-style Relevance Eval (WORKING Colab)
# ROUGE-1 (F-measure) + BERTScore (Recall, IDF)
# =========================================

pred_path = f"{RUNS_DIR}/Smolvlm_Caption.csv"
gt_path   = f"{CAPTIONS_DIR}/test_captions.csv"

!pip -q install bert-score rouge-score

import re, string
import numpy as np
import pandas as pd
from bert_score import score as bertscore
from rouge_score import rouge_scorer

# Load CSVs
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

print("pred_df cols:", pred_df.columns.tolist())
print("gt_df cols  :", gt_df.columns.tolist())

# normalize column names
pred_df.columns = [c.strip().lower() for c in pred_df.columns]
gt_df.columns   = [c.strip().lower() for c in gt_df.columns]

# rename
pred_df = pred_df.rename(columns={"id": "id", "caption": "pred_caption"})
gt_df   = gt_df.rename(columns={"id": "id", "caption": "gt_caption"})

assert {"id", "pred_caption"}.issubset(pred_df.columns)
assert {"id", "gt_caption"}.issubset(gt_df.columns)

# Normalize IDs
def normalize_id(x):
    x = str(x).strip()
    x = re.sub(r"\.(jpg|jpeg|png|bmp|tif|tiff|webp)$", "", x, flags=re.I)
    return x.lower()

pred_df["id_norm"] = pred_df["id"].map(normalize_id)
gt_df["id_norm"]   = gt_df["id"].map(normalize_id)

# Merge
df = pd.merge(
    gt_df[["id_norm", "gt_caption"]],
    pred_df[["id_norm", "pred_caption"]],
    on="id_norm",
    how="inner"
).dropna(subset=["gt_caption", "pred_caption"]).reset_index(drop=True)

print("Merged rows:", len(df))

# CLEF preprocessing
punct_table = str.maketrans("", "", string.punctuation)

def clef_preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"\d+(\.\d+)?", "number", text)
    text = text.translate(punct_table)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["gt_pp"]   = df["gt_caption"].map(clef_preprocess)
df["pred_pp"] = df["pred_caption"].map(clef_preprocess)

# ROUGE-1 F-measure (CLEF exact)
scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=False)
rouge1_f1 = float(np.mean([
    scorer.score(ref, pred)["rouge1"].fmeasure
    for ref, pred in zip(df["gt_pp"].astype(str), df["pred_pp"].astype(str))
]))

# BERTScore Recall + IDF (CLEF-style, Colab compatible)
P, R, F = bertscore(
    cands=df["pred_pp"].astype(str).tolist(),
    refs=df["gt_pp"].astype(str).tolist(),
    model_type="microsoft/deberta-xlarge-mnli",
    lang="en",
    idf=True,
    batch_size=16,
    verbose=True
)

bertscore_recall_idf = float(R.mean().item())

print("\n================ RESULTS (CLEF-style) ================\n")
print(f"ROUGE-1 (F1):              {rouge1_f1:.6f}")
print(f"BERTScore (Recall, IDF):   {bertscore_recall_idf:.6f}")

out_path = f"{WORK_DIR}/clef_eval_merged.csv"
df.to_csv(out_path, index=False)
print("\nSaved merged eval file:", out_path)



pred_df cols: ['ID', 'Caption']
gt_df cols  : ['ID', 'Caption']
Merged rows: 3316
preparing IDF dict...
done in 1.45 seconds
calculating scores...
computing bert embedding.


  0%|          | 0/318 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/208 [00:00<?, ?it/s]

done in 46.27 seconds, 71.67 sentences/sec

================ RESULTS (CLEF-style) ================

ROUGE-1 (F1):              0.496123
BERTScore (Recall, IDF):   0.620543

Saved merged eval file: /content/clef_eval_merged.csv


In [ ]:
!pip -q install bert-score

import pandas as pd, re, os, torch, string
import numpy as np
from bert_score import score
from IPython.display import display

# ===== Paths (EDIT)
pred_path     = f"{RUNS_DIR}/Smolvlm_Caption.csv"
keywords_path = f"{WORK_DIR}/test_with_keywords.csv"   # must have: ID, keywords

# ===== Helpers
def clean_id(x:str)->str:
    x = os.path.basename(str(x))
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.strip().lower()

punct_table = str.maketrans("", "", string.punctuation)

def clef_preproc(s:str)->str:
    """CLEF-style: lowercase, numbers->'number', remove punctuation, squeeze spaces."""
    s = str(s).lower()
    s = re.sub(r"\d+(\.\d+)?", "number", s)
    s = s.translate(punct_table)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_simple(s:str):
    return set(re.findall(r"[a-z0-9]+", clef_preproc(s)))

# ===== Load
pred_df = pd.read_csv(pred_path)
kw_df   = pd.read_csv(keywords_path)

# Normalize columns
pred_df.columns = pred_df.columns.str.strip().str.lower()
kw_df.columns   = kw_df.columns.str.strip().str.lower()

# Rename
pred_df = pred_df.rename(columns={"id":"id_pred", "caption":"caption_pred"})
kw_df   = kw_df.rename(columns={"id":"id_kw"})

if "keywords" not in kw_df.columns:
    raise ValueError("keywords file must contain a 'keywords' column.")

# Clean IDs
pred_df["key"] = pred_df["id_pred"].apply(clean_id)
kw_df["key"]   = kw_df["id_kw"].apply(clean_id)

# Merge
merged = pred_df.merge(kw_df[["key","keywords"]], on="key", how="inner")
print("Merged rows:", len(merged))
display(merged.head(3)[["key","caption_pred","keywords"]])

# CLEF preprocessing
preds_pp = merged["caption_pred"].astype(str).map(clef_preproc).tolist()
keys_pp  = merged["keywords"].astype(str).map(clef_preproc).tolist()

# ===== CLEF-style BERTScore: Recall + IDF
# NOTE: your bert-score version doesn't support idf_sents, so we use idf=True (CLEF-style)
P, R, F1 = score(
    preds_pp, keys_pp,
    model_type="microsoft/deberta-xlarge-mnli",  # CLEF uses this for BERTScore
    lang="en",
    idf=True,
    batch_size=16,
    rescale_with_baseline=False,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True
)

merged["bertscore_recall_kw_clef"] = R.tolist()
merged["bertscore_f1_kw_clef"]     = F1.tolist()

# Exact keyword coverage (token overlap)
cov_scores = []
for cap, kws in zip(merged["caption_pred"].astype(str), merged["keywords"].astype(str)):
    cap_tok = tokenize_simple(cap)
    kw_tok  = tokenize_simple(kws)
    cov_scores.append(0.0 if not kw_tok else len(cap_tok & kw_tok)/len(kw_tok))

merged["exact_keyword_coverage"] = cov_scores

print("\n=== Caption vs Keywords (CLEF-style BERTScore) ===")
print(f"BERTScore Recall+IDF (avg): {merged['bertscore_recall_kw_clef'].mean():.4f}")
print(f"Exact keyword coverage (avg): {merged['exact_keyword_coverage'].mean():.4f}")

# Save
out_csv = f"{RUNS_DIR}/smolvlm_caption_vs_keywords_CLEF_BERTScore.csv"
merged.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


Merged rows: 3316


                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a skin lesion with a reddish-b...   
2  The image shows a chronic inflammatory skin co...   

                                            keywords  
0  ["'cheek papules'", "'clusters'", "'flesh-colo...  
1  ["'allergen'", "'allergic reaction'", "'contac...  
2  ["'cheek papules'", "'clusters'", "'flesh-colo...  

preparing IDF dict...
done in 1.24 seconds
calculating scores...
computing bert embedding.


  0%|          | 0/121 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/208 [00:00<?, ?it/s]

done in 18.06 seconds, 183.66 sentences/sec

=== Caption vs Keywords (CLEF-style BERTScore) ===
BERTScore Recall+IDF (avg): 0.5237
Exact keyword coverage (avg): 0.1831

Saved: /content/drive/MyDrive/SmolVLM_runs/smolvlm_caption_vs_keywords_CLEF_BERTScore.csv


In [ ]:
import pandas as pd

bert_path = f"{RUNS_DIR}/smolvlm_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# merge key
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# filter invalid
merged = merged[merged["fitzpatrick_scale"] > 0].copy()

def tone_group(t):
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)
print("\nTone counts:")
print(merged["tone_group"].value_counts())

tone_summary = (
    merged.groupby("tone_group")[["bertscore_recall_kw_clef","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone (CLEF-style) ===")
print(tone_summary)

out_path = f"{WORK_DIR}/bertscore_kw_clef_with_tone.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)


Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Tone counts:
tone_group
Light     1584
Medium    1168
Dark       457
Name: count, dtype: int64

=== Average Caption Quality by Skin Tone (CLEF-style) ===
  tone_group  bertscore_recall_kw_clef  exact_keyword_coverage
0       Dark                  0.532783                0.210757
1      Light                  0.519452                0.165007
2     Medium                  0.525495                0.197534

Saved merged dataset: /content/bertscore_kw_clef_with_tone.csv


Inference

In [ ]:
from google.colab import drive
drive.mount(f"{WORK_DIR}/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# =========================
# SmolVLM inference (folder-only)
# =========================

!pip -q install -U transformers accelerate bitsandbytes

import os
import torch
import pandas as pd
from PIL import Image
from transformers import AutoProcessor, Idefics3ForConditionalGeneration

# -------------------------
# PATHS
# -------------------------
TEST_IMG_DIR = f"{EVAL_DIR}/Test-image"
OUT_CSV = f"{EVAL_DIR}/smolvlm_test_predictions.csv"

PROMPT = "What do you see here?"

# -------------------------
# LOAD MODEL
# -------------------------
model_id = "HuggingFaceTB/SmolVLM-500M-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = Idefics3ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto"
)
model.eval()

# -------------------------
# HELPER
# -------------------------
@torch.no_grad()
def generate_caption(image: Image.Image, prompt: str, max_new_tokens=96):
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image"}
        ]
    }]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt",
        padding=True
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    caption = processor.batch_decode(
        generated_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )[0].strip()

    return caption

# -------------------------
# RUN INFERENCE
# -------------------------
rows = []
img_files = sorted([
    f for f in os.listdir(TEST_IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
])

for i, fn in enumerate(img_files):
    img_path = os.path.join(TEST_IMG_DIR, fn)
    img_id = os.path.splitext(fn)[0]

    try:
        image = Image.open(img_path).convert("RGB")
        caption = generate_caption(image, PROMPT)
    except Exception as e:
        caption = ""
        print(f"[ERROR] {fn}: {e}")

    rows.append({"ID": img_id, "Caption": caption})

    if i < 5:
        print(f"{img_id} | {caption}")

pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV)
print("Total images processed:", len(rows))



Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

0012821d6f11b96cf33f2c2ee5c68d1f | Close up of a human eye with a small black dot in the corner.
001d22ff2543f95d2d38c18da0446c84 | Red spots on the skin of a person.
002714e65a78f16fb05bc0aa95ea9761 | The image shows a close-up view of a human hand with a visible texture. The skin appears to be dry and slightly wrinkled, with visible lines and creases that suggest aging. The texture is coarse, with visible lines that may indicate the presence of wrinkles or other signs of aging. The skin is also slightly tanned, which is a common feature in older individuals.

The background of the image is blurred, which helps to focus the viewer's attention on the
003e6abf20d234221a41b528b946e90c | The skin of a person has red, blotchy spots.
005b804472c7a27908f99e3d6d6cf91c | A close-up view of a person's head shows multiple red, raised, and inflamed areas.

Saved: /content/drive/MyDrive/Fitz/smolvlm_test_predictions.csv
Total images processed: 3316


In [ ]:
!pip -q install "pandas==2.2.2" "numpy==2.1.3"
import os, sys
print("Restart runtime now: Runtime > Restart runtime")

Restart runtime now: Runtime > Restart runtime


In [ ]:
!pip -q install \
  bert-score==0.3.13 \
  transformers==4.41.2 \
  tokenizers==0.19.1 \
  evaluate==0.4.2 \
  rouge-score==0.1.2


In [ ]:
!pip -q install evaluate bert-score rouge-score

In [ ]:
# =========================================
# 0) Paths
# =========================================
pred_path = f"{EVAL_DIR}/smolvlm_test_predictions.csv"
gt_path   = f"{EVAL_DIR}/test_captions.csv"

# =========================================
# 1) Install only what we need (NO -U)
# =========================================
#!pip -q install evaluate bert-score rouge-score

import re, string
import numpy as np
import pandas as pd
from evaluate import load as hf_load
from bert_score import score as bertscore

# =========================================
# 2) Load CSVs
# =========================================
pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

print("pred_df cols:", pred_df.columns.tolist())
print("gt_df cols  :", gt_df.columns.tolist())

# normalize column names
pred_df.columns = [c.strip().lower() for c in pred_df.columns]
gt_df.columns   = [c.strip().lower() for c in gt_df.columns]

# rename to standard
pred_df = pred_df.rename(columns={"id": "id", "caption": "pred_caption"})
gt_df   = gt_df.rename(columns={"id": "id", "caption": "gt_caption"})

assert {"id", "pred_caption"}.issubset(pred_df.columns), "Pred must have id & caption"
assert {"id", "gt_caption"}.issubset(gt_df.columns), "GT must have ID/Caption (any case)"

# =========================================
# 3) Normalize IDs (THIS usually fixes merge=0)
# =========================================
def normalize_id(x):
    x = str(x).strip()
    # remove common extensions if present
    x = re.sub(r"\.(jpg|jpeg|png|bmp|tif|tiff)$", "", x, flags=re.IGNORECASE)
    return x

pred_df["id_norm"] = pred_df["id"].map(normalize_id)
gt_df["id_norm"]   = gt_df["id"].map(normalize_id)

# Diagnostics: check overlap
pred_ids = set(pred_df["id_norm"].unique())
gt_ids   = set(gt_df["id_norm"].unique())
overlap  = pred_ids.intersection(gt_ids)

print("\nUnique pred ids:", len(pred_ids))
print("Unique gt ids  :", len(gt_ids))
print("Overlap ids    :", len(overlap))

print("\nExample pred ids:", list(pred_ids)[:5])
print("Example gt ids  :", list(gt_ids)[:5])

# show some ids that don't match
print("\nSome pred-only ids:", list(pred_ids - gt_ids)[:10])
print("Some gt-only ids  :", list(gt_ids - pred_ids)[:10])

# =========================================
# 4) Merge using normalized ids
# =========================================
df = pd.merge(
    gt_df[["id_norm", "gt_caption"]],
    pred_df[["id_norm", "pred_caption"]],
    on="id_norm",
    how="inner"
).dropna(subset=["gt_caption", "pred_caption"]).reset_index(drop=True)

print("\nMerged rows:", len(df))
if len(df) == 0:
    raise ValueError("Merged rows is 0. Your IDs still don't match. Check the printed diagnostics above.")

# =========================================
# 5) CLEF-style preprocessing
# =========================================
punct_table = str.maketrans("", "", string.punctuation)

def clef_preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"\d+(\.\d+)?", "number", text)
    text = text.translate(punct_table)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["gt_pp"]   = df["gt_caption"].map(clef_preprocess)
df["pred_pp"] = df["pred_caption"].map(clef_preprocess)

# =========================================
# 6) ROUGE-1 (F1)
# =========================================
rouge = hf_load("rouge")
rouge_res = rouge.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist(),
    rouge_types=["rouge1"]
)
rouge1_f1 = float(rouge_res["rouge1"])

# =========================================
# 7) BERTScore (Recall, IDF)
# =========================================
P, R, F = bertscore(
    cands=df["pred_pp"].tolist(),
    refs=df["gt_pp"].tolist(),
    model_type="microsoft/deberta-xlarge-mnli",
    lang="en",
    idf=True,
    batch_size=16,
    verbose=True
)
bertscore_recall = float(R.mean().item())

# =========================================
# 8) Print results
# =========================================
print("\n================ RESULTS ================\n")
print(f"ROUGE-1 (F1):              {rouge1_f1:.6f}")
print(f"BERTScore (Recall, IDF):   {bertscore_recall:.6f}")

# =========================================
# 9) Save merged + preprocessed file
# =========================================
out_path = f"{WORK_DIR}/clef_eval_merged.csv"
df.to_csv(out_path, index=False)
print("\nSaved merged eval file:", out_path)


pred_df cols: ['ID', 'Caption']
gt_df cols  : ['ID', 'Caption']

Unique pred ids: 3316
Unique gt ids  : 3316
Overlap ids    : 3316

Example pred ids: ['4993c3888dc9586c1c0b0f02f566ac04', 'ee53516bfd29e8559d7db51ffe1c1d62', '20b3c235b9af196c17c0a0c67c8516b0', '2a10c3a61da65ed775ea0e5b7c767472', '9083e3bbb79f974801125a886a7e7e20']
Example gt ids  : ['4993c3888dc9586c1c0b0f02f566ac04', 'ee53516bfd29e8559d7db51ffe1c1d62', '20b3c235b9af196c17c0a0c67c8516b0', '2a10c3a61da65ed775ea0e5b7c767472', '9083e3bbb79f974801125a886a7e7e20']

Some pred-only ids: []
Some gt-only ids  : []

Merged rows: 3316
preparing IDF dict...
done in 1.26 seconds
calculating scores...
computing bert embedding.


  0%|          | 0/392 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/208 [00:00<?, ?it/s]

done in 48.66 seconds, 68.15 sentences/sec

================ RESULTS ================

ROUGE-1 (F1):              0.155288
BERTScore (Recall, IDF):   0.435014

Saved merged eval file: /content/clef_eval_merged.csv


In [ ]:
!pip -q install git+https://github.com/google-research/bleurt.git


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import pandas as pd
import numpy as np
from evaluate import load as hf_load

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")  # your merged file

bleurt = hf_load("bleurt", checkpoint="BLEURT-20")

bleurt_scores = bleurt.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist()
)["scores"]

df["bleurt20"] = bleurt_scores
print("BLEURT-20 (avg):", round(float(np.mean(bleurt_scores)), 6))

df.to_csv(f"{WORK_DIR}/clef_eval_merged_plus_bleurt.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_merged_plus_bleurt.csv")



BLEURT-20 (avg): -1.42676
Saved: /content/clef_eval_merged_plus_bleurt.csv


In [ ]:
# ===============================
# BLEURT-20 (NO MINUS reporting)
# ===============================

!pip -q install git+https://github.com/google-research/bleurt.git
!pip -q install -q evaluate

import pandas as pd
import numpy as np
from evaluate import load as hf_load

# 1) Load merged file you already saved
df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")

# safety
assert "pred_pp" in df.columns and "gt_pp" in df.columns, "Missing pred_pp / gt_pp in clef_eval_merged.csv"

# 2) BLEURT-20 raw (can be negative, that's normal)
bleurt = hf_load("bleurt", checkpoint="BLEURT-20")
bleurt_raw = bleurt.compute(
    predictions=df["pred_pp"].tolist(),
    references=df["gt_pp"].tolist()
)["scores"]

df["bleurt20_raw"] = bleurt_raw

raw_mean = float(np.mean(bleurt_raw))
raw_min  = float(np.min(bleurt_raw))
raw_max  = float(np.max(bleurt_raw))

print("BLEURT-20 raw mean:", round(raw_mean, 6))
print("BLEURT-20 raw min :", round(raw_min, 6))
print("BLEURT-20 raw max :", round(raw_max, 6))

# 3) NO-MINUS version A: shifted to be >= 0
#    (smallest becomes 0)
df["bleurt20_shifted"] = df["bleurt20_raw"] - raw_min
shifted_mean = float(df["bleurt20_shifted"].mean())

print("\nBLEURT-20 shifted mean (>=0):", round(shifted_mean, 6))
print("Shifted min:", round(float(df['bleurt20_shifted'].min()), 6))

# 4) NO-MINUS version B (recommended): normalize to [0, 1]
#    (best for averaging with ROUGE/BERTScore)
den = (raw_max - raw_min) if (raw_max - raw_min) != 0 else 1e-12
df["bleurt20_norm01"] = (df["bleurt20_raw"] - raw_min) / den
norm_mean = float(df["bleurt20_norm01"].mean())

print("\nBLEURT-20 normalized [0,1] mean:", round(norm_mean, 6))
print("Norm min:", round(float(df['bleurt20_norm01'].min()), 6),
      "Norm max:", round(float(df['bleurt20_norm01'].max()), 6))

# 5) Save
out_path = f"{WORK_DIR}/clef_eval_merged_plus_bleurt_nominas.csv"
df.to_csv(out_path, index=False)
print("\nSaved:", out_path)

# If you want a single BLEURT number with no minus for reporting, use:
print("\nREPORT THIS (no minus): BLEURT-20_norm01 =", round(norm_mean, 6))



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


BLEURT-20 raw mean: -1.42676
BLEURT-20 raw min : -2.065816
BLEURT-20 raw max : -0.133911

BLEURT-20 shifted mean (>=0): 0.639056
Shifted min: 0.0

BLEURT-20 normalized [0,1] mean: 0.330791
Norm min: 0.0 Norm max: 1.0

Saved: /content/clef_eval_merged_plus_bleurt_nominas.csv

REPORT THIS (no minus): BLEURT-20_norm01 = 0.330791


In [ ]:
!pip -q install transformers accelerate --no-deps


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged_plus_bleurt_nominas.csv")

premises   = df["gt_pp"].astype(str).tolist()     # reference/context
hypotheses = df["pred_pp"].astype(str).tolist()   # claim/prediction

model_name = "microsoft/deberta-large-mnli"  # ✅ correct model (no 404)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

id2label = {int(k): v for k, v in model.config.id2label.items()}
print("id2label:", id2label)

# find entailment index robustly
entail_idx = None
for k, v in id2label.items():
    if str(v).lower().startswith("entail"):
        entail_idx = k
        break
if entail_idx is None:
    entail_idx = 2  # common MNLI ordering

def batch_entailment(premises, hypotheses, batch_size=16, max_len=256):
    scores = []
    for i in range(0, len(premises), batch_size):
        p = premises[i:i+batch_size]
        h = hypotheses[i:i+batch_size]
        enc = tokenizer(
            p, h, truncation=True, padding=True, max_length=max_len,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        scores.extend(probs[:, entail_idx].tolist())
    return np.array(scores)

ent_scores = batch_entailment(premises, hypotheses, batch_size=16, max_len=256)
df["nli_align_entail"] = ent_scores

print("NLI-Align (Entailment prob) avg:", round(float(ent_scores.mean()), 6))
print("min:", round(float(ent_scores.min()), 6), "max:", round(float(ent_scores.max()), 6))

out_path = f"{WORK_DIR}/clef_eval_with_nli_align.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


id2label: {0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}
NLI-Align (Entailment prob) avg: 0.142718
min: 8.9e-05 max: 0.994888
Saved: /content/clef_eval_with_nli_align.csv


In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter

# ---------------------------
# 0) Load your existing eval file
# ---------------------------
df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_nli_align.csv")

# You already have these two as constants from your earlier run:
rouge1_f1 = 0.155288
bertscore_recall_idf = 0.435014


# BLEURT normalized (0..1, no minus) should already exist from your BLEURT cell
assert "bleurt20_norm01" in df.columns, "Missing bleurt20_norm01. Run BLEURT no-minus code first."
bleurt_norm = float(df["bleurt20_norm01"].mean())

# NLI align entailment (0..1)
assert "nli_align_entail" in df.columns, "Missing nli_align_entail. Run NLI-align code first."
nli_align = float(df["nli_align_entail"].mean())

print("Loaded rows:", len(df))


# ---------------------------
# 1) UMLS Concept F1 (best-effort)
#    - Tries scispaCy UMLS linker first.
#    - If it fails, falls back to a lightweight "medical-term concept" F1 (not true UMLS).
# ---------------------------

def f1_from_sets(pred_set, ref_set):
    pred_set = set(pred_set)
    ref_set = set(ref_set)
    if len(pred_set) == 0 and len(ref_set) == 0:
        return 1.0
    if len(pred_set) == 0 or len(ref_set) == 0:
        return 0.0
    tp = len(pred_set & ref_set)
    fp = len(pred_set - ref_set)
    fn = len(ref_set - pred_set)
    prec = tp / (tp + fp + 1e-12)
    rec  = tp / (tp + fn + 1e-12)
    return 2 * prec * rec / (prec + rec + 1e-12)

umls_mode = None

try:
    # Install only if needed (comment out if already installed)
    !pip -q install spacy scispacy
    !pip -q install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz

    import spacy
    from scispacy.linking import UmlsEntityLinker

    nlp = spacy.load("en_core_sci_md")
    linker = UmlsEntityLinker(resolve_abbreviations=True, name="umls")
    nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})

    def extract_cuis(text: str):
        doc = nlp(text)
        cuis = []
        for ent in doc.ents:
            for kb_ent in ent._.kb_ents:
                cui = kb_ent[0]  # CUI string
                cuis.append(cui)
        return set(cuis)

    # Compute per-sample UMLS F1
    f1s = []
    for gt, pred in zip(df["gt_pp"].astype(str), df["pred_pp"].astype(str)):
        gt_cuis = extract_cuis(gt)
        pr_cuis = extract_cuis(pred)
        f1s.append(f1_from_sets(pr_cuis, gt_cuis))

    df["umls_f1"] = f1s
    umls_f1_avg = float(np.mean(f1s))
    umls_mode = "UMLS_CUI_F1 (scispaCy linker)"

except Exception as e:
    # Fallback: NOT true UMLS, but still a concept-like term overlap F1
    # Useful if UMLS resources aren't available in Colab.
    MED_TERMS = set([
        "macule","papule","plaque","patch","nodule","vesicle","pustule",
        "ulcer","erosion","crust","scale","erythema","hyperpigmentation",
        "hypopigmentation","melanoma","nevus","lesion","tumor","benign","malignant",
        "asymmetry","border","color","diameter","evolution","itch","bleeding"
    ])

    def extract_terms(text: str):
        toks = re.findall(r"[a-z]+", text.lower())
        return set([t for t in toks if t in MED_TERMS])

    f1s = []
    for gt, pred in zip(df["gt_pp"].astype(str), df["pred_pp"].astype(str)):
        gt_terms = extract_terms(gt)
        pr_terms = extract_terms(pred)
        f1s.append(f1_from_sets(pr_terms, gt_terms))

    df["umls_f1"] = f1s
    umls_f1_avg = float(np.mean(f1s))
    umls_mode = "Fallback Term-F1 (NOT true UMLS)"
    print("\n[WARN] True UMLS linking failed in this runtime.")
    print("Using fallback term-based concept F1 instead.")
    print("Reason (first 200 chars):", str(e)[:200])


# ---------------------------
# 2) Image–Caption Similarity (optional)
#    - Requires an image path column in df
#    - If you have images, set IMAGE_COL to your column name.
# ---------------------------
IMAGE_COL = None  # e.g., "image_path"  (set this if you have it)

sim_avg = None
if IMAGE_COL is not None and IMAGE_COL in df.columns:
    !pip -q install open_clip_torch pillow

    import torch
    import open_clip
    from PIL import Image

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # A practical CLIP baseline (not medical-specific). If you want BioMedCLIP later, tell me.
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    model = model.to(device).eval()

    def clip_similarity(image_path, caption):
        try:
            img = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
        except:
            return np.nan
        text = tokenizer([caption]).to(device)
        with torch.no_grad():
            img_feat = model.encode_image(img)
            txt_feat = model.encode_text(text)
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
            txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
            sim = (img_feat @ txt_feat.T).squeeze().item()
        return sim

    sims = []
    for p, cap in zip(df[IMAGE_COL].astype(str), df["pred_pp"].astype(str)):
        sims.append(clip_similarity(p, cap))

    df["img_caption_sim"] = sims
    sim_avg = float(np.nanmean(sims))
else:
    print("\n[INFO] Image–Caption similarity skipped (no IMAGE_COL set / not present).")


# ---------------------------
# 3) Compute the requested averages
# ---------------------------

# Relevance metrics: ROUGE, BERTScore, BLEURT_norm, Similarity
# If similarity wasn't computed, we compute relevance over the available 3.
relevance_parts = [
    ("ROUGE-1_F1", rouge1_f1),
    ("BERTScore_Recall_IDF", bertscore_recall_idf),
    ("BLEURT20_norm01", bleurt_norm),
]

if sim_avg is not None:
    relevance_parts.append(("ImgCaptionSimilarity", sim_avg))

relevance_avg = float(np.mean([v for _, v in relevance_parts]))

# Factuality metrics: UMLS_F1 + NLI-align
factuality_avg = float(np.mean([umls_f1_avg, nli_align]))

# Overall: average of relevance_avg and factuality_avg (clean 2-aspect CLEF-style)
overall_score = float(np.mean([relevance_avg, factuality_avg]))

print("\n==================== FINAL REPORT ====================\n")

print("Relevance metrics:")
for k, v in relevance_parts:
    print(f"  {k}: {v:.6f}")
print(f"  Relevance average: {relevance_avg:.6f}")

print("\nFactuality metrics:")
print(f"  UMLS Concept F1 (avg): {umls_f1_avg:.6f}   [{umls_mode}]")
print(f"  NLI-Align entail (avg): {nli_align:.6f}")
print(f"  Factuality average: {factuality_avg:.6f}")

print("\nOverall:")
print(f"  Overall score (avg of relevance & factuality): {overall_score:.6f}")

# Save a final file
out_path = f"{WORK_DIR}/clef_eval_final_report.csv"
df.to_csv(out_path, index=False)
print("\nSaved per-sample file:", out_path)


Loaded rows: 3316
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[WARN] True UMLS linking failed in this runtime.
Using fallback term-based concept F1 instead.
Reason (first 200 chars): numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

[INFO] Image–Caption similarity skipped (no IMAGE_COL set / not present).

==================== FINAL REPORT ====================

Relevance metrics:
  ROUGE-1_F1: 0.155288
  BERTScore_Recall_IDF: 0.435014
  BLEURT20_norm01: 0.330791
  Relevance average: 0.307031

Factuality metrics:
  UMLS Concept F1 (avg): 0.192742   [Fallback Term-F1 (NOT true UMLS)]
  NLI-Align entail (avg): 0.142718
  Factuality average: 0.167730

Overall:
  Overall score (avg of relevance & factuality): 0.237381

Saved per-sample file: /content/clef_eval_final_report.csv


In [ ]:
!pip -q install "transformers==4.44.2" --no-deps


In [ ]:
import os

IMG_ROOT = f"{EVAL_DIR}/Test-image"

print("Exists?", os.path.exists(IMG_ROOT))
print("Top-level items:", os.listdir(IMG_ROOT)[:30] if os.path.exists(IMG_ROOT) else "NOT FOUND")


Exists? True
Top-level items: ['219a8699982a865eaf4d8a0e10f011c8.jpg', '6c832de023a694cdbc102b2ce9e22362.jpg', '85a850210bbeb9ae8f5f33d161884ed3.jpg', 'ecc93a4b17e1209ddac406c31f2794c1.jpg', '1e7c9106644d4b54d11524d75d303a35.jpg', 'c55612ae428726d00c2539a00763f1c5.jpg', '8f33d04e000dda8a5c0785f95693d6dc.jpg', 'a741bce58cf478035b530343f1bd0646.jpg', 'f12b08c765518b9d3e60ea5cc9dfdaa4.jpg', '600027ed492ec1c0835a06e9f2586f2c.jpg', '221237ebdea54eea0049d29291a2c918.jpg', 'ca33407ab1e504ec6249e22d0436490a.jpg', '2ddc13cb0aad47b2dda40db94427db99.jpg', 'f5e0a084eaf8cfcfb94c4093f9a31e48.jpg', '572a802d11a61721053e7dee8911cfa6.jpg', '8c314e97a1c322f6949f27d68356e0ee.jpg', '656c560e200b0cfca63346939fa848d3.jpg', '75a4dac11ce8c24d40654d680eb3eb05.jpg', '52907d7a88da7fa4b6097357a05c1413.jpg', '16b8bc3e9b110be6a2a9fd20ae126dc3.jpg', 'a0761e1b5f6eacc88e95ee6f871427a1.jpg', 'f43f1d220c2061be70ccce775c095c1e.jpg', 'f563de4c84fed5c664de85aa37d72f16.jpg', '1fa3de789800454bff91bcae5bf99599.jpg', 'd4851632

In [ ]:
import os, glob
import pandas as pd

pred_path = f"{EVAL_DIR}/smolvlm_test_predictions.csv"
gt_path   = f"{EVAL_DIR}/test_captions.csv"

pred_df = pd.read_csv(pred_path)
gt_df   = pd.read_csv(gt_path)

print("pred_df cols:", pred_df.columns.tolist())
print("gt_df cols  :", gt_df.columns.tolist())


pred_df cols: ['ID', 'Caption']
gt_df cols  : ['ID', 'Caption']


In [ ]:
import pandas as pd, numpy as np

rouge1_f1 = 0.155288
bertscore_recall = 0.435014



df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_nli_align.csv")

bleurt_norm = float(df["bleurt20_norm01"].mean())   # 0..1
nli_align   = float(df["nli_align_entail"].mean())  # 0..1

final_avg_4 = float(np.mean([rouge1_f1, bertscore_recall, bleurt_norm, nli_align]))

print("\n===== FINAL SCORE (4 metrics, no minus) =====")
print(f"ROUGE-1 (F1):               {rouge1_f1:.6f}")
print(f"BERTScore (Recall, IDF):    {bertscore_recall:.6f}")
print(f"BLEURT-20_norm01 (avg):     {bleurt_norm:.6f}")
print(f"NLI-Align Entail (avg):     {nli_align:.6f}")
print(f"\nAverage over 4 metrics:     {final_avg_4:.6f}")



===== FINAL SCORE (4 metrics, no minus) =====
ROUGE-1 (F1):               0.155288
BERTScore (Recall, IDF):    0.435014
BLEURT-20_norm01 (avg):     0.330791
NLI-Align Entail (avg):     0.142718

Average over 4 metrics:     0.265953


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

# ---------------------------
# 1) FILES
# ---------------------------
train_kw_path = f"{WORK_DIR}/train.csv"   # 11k+
test_eval_path = f"{WORK_DIR}/clef_eval_merged.csv"                 # has ID + caption_true/pred
test_label_path = f"{WORK_DIR}/test.csv"   # has ID + label_name (common label)

# ---------------------------
# 2) COLUMNS (CHANGE THESE)
# ---------------------------
TRAIN_ID_COL = "image_id"
TRAIN_LABEL_COL = "label"     # common label name
TRAIN_KW_COL = "concepts"          # could be "keyword" or list-like string

TEST_ID_COL = "md5hash"
TEST_LABEL_COL = "label"

GT_CAP_COL = "gt_caption"
PRED_CAP_COL = "pred_caption"

# ---------------------------
# 3) Helpers
# ---------------------------
def norm_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_keywords(x):
    """Handles: 'k1;k2', 'k1, k2', "['k1','k2']", etc."""
    if pd.isna(x): return []
    s = str(x).strip()
    # list-like
    if s.startswith("[") and s.endswith("]"):
        s = s.strip("[]")
    # split by common separators
    parts = re.split(r"[;,|]\s*|,\s*", s)
    parts = [p.strip().lower() for p in parts if p.strip()]
    return parts

def build_patterns(vocab):
    vocab = sorted(set([v.strip().lower() for v in vocab if len(str(v).strip()) >= 2]),
                   key=len, reverse=True)
    return [(k, re.compile(rf"(?<!\w){re.escape(k)}(?!\w)")) for k in vocab]

def extract_from_vocab(text, patterns):
    t = norm_text(text)
    found = set()
    for k, pat in patterns:
        if pat.search(t):
            found.add(k)
    return found

def f1_from_sets(pred_set, gt_set):
    if len(pred_set)==0 and len(gt_set)==0:
        return 1.0
    if len(pred_set)==0 or len(gt_set)==0:
        return 0.0
    tp = len(pred_set & gt_set)
    p = tp/len(pred_set) if pred_set else 0.0
    r = tp/len(gt_set) if gt_set else 0.0
    return (2*p*r/(p+r)) if (p+r) else 0.0

# ---------------------------
# 4) Load train keywords and build label->keywords map
# ---------------------------
train_df = pd.read_csv(train_kw_path)
train_df.columns = train_df.columns.str.lower().str.strip()

# if your real columns are different, update the names above
train_df = train_df.rename(columns={
    TRAIN_ID_COL.lower(): "image_id",
    TRAIN_LABEL_COL.lower(): "label",
    TRAIN_KW_COL.lower(): "concepts"
})

label_kw_counter = defaultdict(Counter)

for _, row in train_df.iterrows():
    lbl = str(row["label"]).strip().lower()
    kws = split_keywords(row["concepts"])
    for k in kws:
        label_kw_counter[lbl][k] += 1

# keep top-K keywords per label (tune K)
TOPK = 30
label_kw_map = {lbl: set([k for k,_ in cnt.most_common(TOPK)]) for lbl, cnt in label_kw_counter.items()}

print("Labels in train:", len(label_kw_map))
print("Example label keywords:", list(label_kw_map.items())[:1])

# optional: global vocab from all label keywords
global_vocab = set().union(*label_kw_map.values())
patterns = build_patterns(global_vocab)
print("Global keyword vocab size:", len(global_vocab))

# ---------------------------
# 5) Load test labels + eval captions, align by ID
# ---------------------------
test_labels = pd.read_csv(test_label_path)
test_labels.columns = test_labels.columns.str.lower().str.strip()
test_labels = test_labels.rename(columns={TEST_ID_COL.lower():"id", TEST_LABEL_COL.lower():"label_name"})
test_labels["id"] = test_labels["id"].astype(str).str.strip()

eval_df = pd.read_csv(test_eval_path)
eval_df.columns = eval_df.columns.str.lower().str.strip()

# adjust if needed
if "id" not in eval_df.columns and "id_norm" in eval_df.columns:
    eval_df["id"] = eval_df["id_norm"]

eval_df["id"] = eval_df["id"].astype(str).str.strip()

df = eval_df.merge(test_labels[["id","label_name"]], on="id", how="left")
df["label_name"] = df["label_name"].astype(str).str.strip().str.lower()

# ---------------------------
# 6) Metric A: Keyword F1 between GT vs Pred captions (global keyword vocab)
# ---------------------------
gt_sets = df[GT_CAP_COL.lower()].fillna("").astype(str).apply(lambda x: extract_from_vocab(x, patterns))
pr_sets = df[PRED_CAP_COL.lower()].fillna("").astype(str).apply(lambda x: extract_from_vocab(x, patterns))

df["kw_f1_gt_vs_pred"] = [f1_from_sets(p,g) for p,g in zip(pr_sets, gt_sets)]
print("Keyword F1 (GT vs Pred) avg:", round(float(df["kw_f1_gt_vs_pred"].mean()), 6))

# ---------------------------
# 7) Metric B (optional but useful): Label-grounding score
#     Pred keywords vs label-derived keyword set
# ---------------------------
label_sets = df["label_name"].apply(lambda l: label_kw_map.get(l, set()))
df["kw_f1_pred_vs_label"] = [f1_from_sets(p, lab) for p,lab in zip(pr_sets, label_sets)]
print("Keyword F1 (Pred vs Label) avg:", round(float(df["kw_f1_pred_vs_label"].mean()), 6))

df.to_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics.csv", index=False)
print("Saved: /content/clef_eval_with_label_keyword_metrics.csv")


Labels in train: 114
Example label keywords: [('hidradenitis', {"'boil-like nodules.'", "'persistent nodules'", "'recurrent nodules'", "'sinuses'", "'scarring'", "'purulent discharge'", "'abscesses'", "'chronic inflammatory skin condition'"})]
Global keyword vocab size: 791
Keyword F1 (GT vs Pred) avg: 1.0
Keyword F1 (Pred vs Label) avg: 0.0
Saved: /content/clef_eval_with_label_keyword_metrics.csv


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

df = pd.read_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics.csv")

train_kw_path = f"{WORK_DIR}/train.csv"  # <-- put your real file
train_df = pd.read_csv(train_kw_path)
train_df.columns = train_df.columns.str.lower().str.strip()

# change these if needed
train_df = train_df.rename(columns={
    "label": "label_name",
    "concepts": "keywords"
})

def norm_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"\d+", "number", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_keywords(x):
    if pd.isna(x): return []
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s.strip("[]")
    parts = re.split(r"[;,|]\s*|,\s*", s)
    parts = [p.strip() for p in parts if p.strip()]
    return parts

# build label->keyword counter (BUT normalize each keyword!)
label_kw_counter = defaultdict(Counter)

for _, row in train_df.iterrows():
    lbl = norm_text(row["label_name"])
    kws = split_keywords(row["keywords"])
    for k in kws:
        k2 = norm_text(k)      # ✅ normalize keyword
        if len(k2) >= 2:
            label_kw_counter[lbl][k2] += 1

TOPK = 30
label_kw_map = {lbl: set([k for k,_ in cnt.most_common(TOPK)]) for lbl, cnt in label_kw_counter.items()}

global_vocab = set().union(*label_kw_map.values())

print("Labels:", len(label_kw_map))
print("Global vocab:", len(global_vocab))
print("Example normalized keywords:", list(label_kw_map.items())[:1])

# regex patterns from normalized keywords
vocab_sorted = sorted(global_vocab, key=len, reverse=True)
patterns = [(k, re.compile(rf"(?<!\w){re.escape(k)}(?!\w)")) for k in vocab_sorted]

def extract_vocab(text):
    t = norm_text(text)
    found = set()
    for k, pat in patterns:
        if pat.search(t):
            found.add(k)
    return found

def f1_from_sets(pred_set, gt_set):
    if len(pred_set)==0 and len(gt_set)==0:
        return np.nan  # ✅ IMPORTANT: do NOT give 1.0 for empty-empty
    if len(pred_set)==0 or len(gt_set)==0:
        return 0.0
    tp = len(pred_set & gt_set)
    p = tp/len(pred_set)
    r = tp/len(gt_set)
    return (2*p*r/(p+r)) if (p+r) else 0.0

# detect columns
gt_col = "gt_caption" if "gt_caption" in df.columns else "caption_gt"
pred_col = "pred_caption" if "pred_caption" in df.columns else "pred_caption"

df["kw_set_gt"] = df[gt_col].fillna("").astype(str).apply(extract_vocab)
df["kw_set_pred"] = df[pred_col].fillna("").astype(str).apply(extract_vocab)

df["kw_f1_gt_vs_pred"] = [
    f1_from_sets(p,g) for p,g in zip(df["kw_set_pred"], df["kw_set_gt"])
]

print("\nKeyword F1 (GT vs Pred) avg (ignoring NaN):",
      round(float(np.nanmean(df["kw_f1_gt_vs_pred"])), 6))

# Show how many are empty sets
print("Empty GT keyword sets:", (df["kw_set_gt"].apply(len)==0).sum(), "/", len(df))
print("Empty Pred keyword sets:", (df["kw_set_pred"].apply(len)==0).sum(), "/", len(df))

df.to_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics_FIXED.csv", index=False)
print("Saved: /content/clef_eval_with_label_keyword_metrics_FIXED.csv")


Labels: 114
Global vocab: 784
Example normalized keywords: [('hidradenitis', {'recurrent nodules', 'persistent nodules', 'purulent discharge', 'boil like nodules', 'abscesses', 'chronic inflammatory skin condition', 'sinuses', 'scarring'})]

Keyword F1 (GT vs Pred) avg (ignoring NaN): 0.113044
Empty GT keyword sets: 2 / 3316
Empty Pred keyword sets: 953 / 3316
Saved: /content/clef_eval_with_label_keyword_metrics_FIXED.csv


In [ ]:
!pip -q install open_clip_torch pillow --no-deps


In [ ]:
!pip -q install ftfy regex


In [ ]:
import os
import glob
import pandas as pd

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged.csv")

IMG_ROOT = f"{EVAL_DIR}/Test-image"

# ✅ safer glob (directly search jpg files)
all_imgs = glob.glob(os.path.join(IMG_ROOT, "*.jpg"))
print("Found JPG images:", len(all_imgs))

# Map filename -> path
img_map = {os.path.splitext(os.path.basename(p))[0]: p for p in all_imgs}

df["image_path"] = df["id_norm"].astype(str).map(img_map)

matched = df["image_path"].notna().sum()
print("Matched images:", matched, "/", len(df))

df.to_csv(f"{WORK_DIR}/clef_eval_merged_with_paths.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_merged_with_paths.csv")


Found JPG images: 3316
Matched images: 3316 / 3316
Saved: /content/clef_eval_merged_with_paths.csv


In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
import open_clip

df = pd.read_csv(f"{WORK_DIR}/clef_eval_merged_with_paths.csv")
df_ok = df.dropna(subset=["image_path"]).copy()

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
model, _, preprocess = open_clip.create_model_and_transforms(model_id)
tokenizer = open_clip.get_tokenizer(model_id)
model = model.to(device).eval()

def sim_one(img_path, caption):
    img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    txt = tokenizer([str(caption)]).to(device)

    with torch.no_grad():
        imf = model.encode_image(img)
        txf = model.encode_text(txt)

        imf = imf / imf.norm(dim=-1, keepdim=True)
        txf = txf / txf.norm(dim=-1, keepdim=True)

        return float((imf @ txf.T).squeeze().item())

# use predicted caption text for similarity
sims = []
for p, cap in zip(df_ok["image_path"], df_ok["pred_caption"]):
    sims.append(sim_one(p, cap))

df_ok["img_caption_sim"] = sims
sim_avg = float(np.mean(sims))

print("✅ Image–Caption Similarity avg (BiomedCLIP):", round(sim_avg, 6))

df.loc[df_ok.index, "img_caption_sim"] = df_ok["img_caption_sim"].values
df.to_csv(f"{WORK_DIR}/clef_eval_with_similarity.csv", index=False)
print("Saved:", f"{WORK_DIR}/clef_eval_with_similarity.csv")

open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ Image–Caption Similarity avg (BiomedCLIP): 0.347779
Saved: /content/clef_eval_with_similarity.csv


In [ ]:
import pandas as pd
import numpy as np

# =============================
# Files
# =============================
sim_df  = pd.read_csv(f"{WORK_DIR}/clef_eval_with_similarity.csv")                 # has similarity per id
base_df = pd.read_csv(f"{WORK_DIR}/clef_eval_final_report.csv")                   # has bleurt + nli_align_entail
kw_df   = pd.read_csv(f"{WORK_DIR}/clef_eval_with_label_keyword_metrics_FIXED.csv")  # has kw_f1_gt_vs_pred

# =============================
# Helper: pick a column safely
# =============================
def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# normalize column names (important!)
sim_df.columns  = sim_df.columns.str.strip()
base_df.columns = base_df.columns.str.strip()
kw_df.columns   = kw_df.columns.str.strip()

# =============================
# Detect ID columns
# =============================
id_sim  = pick_first(sim_df,  ["id", "id_norm", "id_true", "ID"])
id_base = pick_first(base_df, ["id", "id_norm", "id_true", "ID"])
id_kw   = pick_first(kw_df,   ["id", "id_norm", "id_true", "ID"])

print("ID cols:", id_sim, id_base, id_kw)

# =============================
# Create merge keys
# =============================
def norm_id(x):
    x = str(x).strip()
    return x.replace(".jpg","").replace(".png","")

sim_df["id_key"]  = sim_df[id_sim].apply(norm_id)
base_df["id_key"] = base_df[id_base].apply(norm_id)
kw_df["id_key"]   = kw_df[id_kw].apply(norm_id)

# =============================
# Detect similarity column
# =============================
SIM_COL = pick_first(sim_df, ["img_caption_sim", "similarity", "sim"])
print("SIM_COL:", SIM_COL)

# merge similarity into base
df = base_df.merge(sim_df[["id_key", SIM_COL]], on="id_key", how="left")

# merge keyword f1 too
KWF1_COL = pick_first(kw_df, ["kw_f1_gt_vs_pred", "kw_f1"])
print("KWF1_COL:", KWF1_COL)

df = df.merge(kw_df[["id_key", KWF1_COL]], on="id_key", how="left")

# =============================
# Use your already computed constants
# =============================
ROUGE1_F1 = 0.155288
BERTSCORE_RECALL = 0.435014

# detect bleurt + align columns in base
BLEURT_COL = pick_first(df, ["bleurt20_norm01", "BLEURT20_norm01", "bleurt"])
ALIGN_COL  = pick_first(df, ["nli_align_entail", "NLI_align_entail", "alignscore", "AlignScore"])

print("BLEURT_COL:", BLEURT_COL)
print("ALIGN_COL :", ALIGN_COL)

BLEURT = float(pd.to_numeric(df[BLEURT_COL], errors="coerce").mean())
ALIGNSCORE = float(pd.to_numeric(df[ALIGN_COL], errors="coerce").mean())
SIM = float(pd.to_numeric(df[SIM_COL], errors="coerce").mean())

# keyword f1 (ignore NaNs)
DERM_KWF1 = float(np.nanmean(pd.to_numeric(df[KWF1_COL], errors="coerce")))

# =============================
# CLEF-like averages
# =============================
RELEVANCE_AVG = float(np.mean([ROUGE1_F1, BERTSCORE_RECALL, BLEURT, SIM]))

# factuality avg WITHOUT UMLS:
# Option 1: Only AlignScore (CLEF-like fallback)
# FACTUALITY_AVG = ALIGNSCORE

# Option 2 (recommended): AlignScore + Derm Keyword Concept F1
FACTUALITY_AVG = float(np.mean([ALIGNSCORE, DERM_KWF1]))

OVERALL = float(np.mean([RELEVANCE_AVG, FACTUALITY_AVG]))

# =============================
# Print CLEF-style row
# =============================
row = {
    "ID": 1,
    "Submission Name": "smolvlm",
    "Overall": round(OVERALL, 6),
    "Similarity": round(SIM, 6),
    "BERTScore (Recall)": round(BERTSCORE_RECALL, 6),
    "ROUGE-1": round(ROUGE1_F1, 6),
    "BLEURT": round(BLEURT, 6),
    "Relevance Average": round(RELEVANCE_AVG, 6),
    "Derm Keyword Concept F1": round(DERM_KWF1, 6),
    "AlignScore": round(ALIGNSCORE, 6),
    "Factuality Average": round(FACTUALITY_AVG, 6),
}

leaderboard = pd.DataFrame([row])
leaderboard




ID cols: id_norm id_norm id
SIM_COL: img_caption_sim
KWF1_COL: kw_f1_gt_vs_pred
BLEURT_COL: bleurt20_norm01
ALIGN_COL : nli_align_entail


   ID Submission Name  Overall  Similarity  BERTScore (Recall)  \
0   1 smolvlm  0.22255    0.347779            0.435014   

    ROUGE-1    BLEURT  Relevance Average  Derm Keyword Concept F1  AlignScore  \
0  0.155288  0.330791           0.317218                 0.113044    0.142718   

   Factuality Average  
0            0.127881  

In [ ]:
!pip -q uninstall -y sentence-transformers
!pip -q uninstall -y bert-score transformers tokenizers
!pip -q install bert-score==0.3.13 transformers==4.38.2 tokenizers==0.15.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.6 MB/s eta 0:00:00


In [ ]:
!pip -q install bert-score

import pandas as pd, re, os, torch, string
import numpy as np
from bert_score import score
from IPython.display import display

# ===== Paths (EDIT)
pred_path     = f"{WORK_DIR}/Smolvlm_Caption (1).csv"
keywords_path = f"{WORK_DIR}/test_with_keywords.csv"   # must have: ID, keywords

# ===== Helpers
def clean_id(x:str)->str:
    x = os.path.basename(str(x))
    x = re.sub(r"\.(jpg|jpeg|png|bmp|gif|tif|tiff|webp)$", "", x, flags=re.I)
    return x.strip().lower()

punct_table = str.maketrans("", "", string.punctuation)

def clef_preproc(s:str)->str:
    """CLEF-style: lowercase, numbers->'number', remove punctuation, squeeze spaces."""
    s = str(s).lower()
    s = re.sub(r"\d+(\.\d+)?", "number", s)
    s = s.translate(punct_table)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_simple(s:str):
    return set(re.findall(r"[a-z0-9]+", clef_preproc(s)))

# ===== Load
pred_df = pd.read_csv(pred_path)
kw_df   = pd.read_csv(keywords_path)

# Normalize columns
pred_df.columns = pred_df.columns.str.strip().str.lower()
kw_df.columns   = kw_df.columns.str.strip().str.lower()

# Rename
pred_df = pred_df.rename(columns={"id":"id_pred", "caption":"caption_pred"})
kw_df   = kw_df.rename(columns={"id":"id_kw"})

if "keywords" not in kw_df.columns:
    raise ValueError("keywords file must contain a 'keywords' column.")

# Clean IDs
pred_df["key"] = pred_df["id_pred"].apply(clean_id)
kw_df["key"]   = kw_df["id_kw"].apply(clean_id)

# Merge
merged = pred_df.merge(kw_df[["key","keywords"]], on="key", how="inner")
print("Merged rows:", len(merged))
display(merged.head(3)[["key","caption_pred","keywords"]])

# CLEF preprocessing
preds_pp = merged["caption_pred"].astype(str).map(clef_preproc).tolist()
keys_pp  = merged["keywords"].astype(str).map(clef_preproc).tolist()

# ===== CLEF-style BERTScore: Recall + IDF
# NOTE: your bert-score version doesn't support idf_sents, so we use idf=True (CLEF-style)
P, R, F1 = score(
    preds_pp, keys_pp,
    model_type="microsoft/deberta-xlarge-mnli",  # CLEF uses this for BERTScore
    lang="en",
    idf=True,
    batch_size=16,
    rescale_with_baseline=False,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True
)

merged["bertscore_recall_kw_clef"] = R.tolist()
merged["bertscore_f1_kw_clef"]     = F1.tolist()

# Exact keyword coverage (token overlap)
cov_scores = []
for cap, kws in zip(merged["caption_pred"].astype(str), merged["keywords"].astype(str)):
    cap_tok = tokenize_simple(cap)
    kw_tok  = tokenize_simple(kws)
    cov_scores.append(0.0 if not kw_tok else len(cap_tok & kw_tok)/len(kw_tok))

merged["exact_keyword_coverage"] = cov_scores

print("\n=== Caption vs Keywords (CLEF-style BERTScore) ===")
print(f"BERTScore Recall+IDF (avg): {merged['bertscore_recall_kw_clef'].mean():.4f}")
print(f"Exact keyword coverage (avg): {merged['exact_keyword_coverage'].mean():.4f}")

# Save
out_csv = f"{WORK_DIR}/smolvlm_caption_vs_keywords_CLEF_BERTScore.csv"
merged.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


Merged rows: 3316


                                key  \
0  0012821d6f11b96cf33f2c2ee5c68d1f   
1  001d22ff2543f95d2d38c18da0446c84   
2  002714e65a78f16fb05bc0aa95ea9761   

                                        caption_pred  \
0  The image shows a close-up of a person's eye; ...   
1  The image shows a skin lesion with a reddish-b...   
2  The image shows a chronic inflammatory skin co...   

                                            keywords  
0  ["'cheek papules'", "'clusters'", "'flesh-colo...  
1  ["'allergen'", "'allergic reaction'", "'contac...  
2  ["'cheek papules'", "'clusters'", "'flesh-colo...  

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

...

preparing IDF dict...
done in 1.19 seconds
calculating scores...
computing bert embedding.


  0%|          | 0/121 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/208 [00:00<?, ?it/s]

done in 23.50 seconds, 141.11 sentences/sec

=== Caption vs Keywords (CLEF-style BERTScore) ===
BERTScore Recall+IDF (avg): 0.5237
Exact keyword coverage (avg): 0.1831

Saved: /content/smolvlm_caption_vs_keywords_CLEF_BERTScore.csv


In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/smolvlm_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# ---- merge key ----
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# ---- fitzpatrick to names (keep -1 as Unknown) ----
merged["fitzpatrick_scale"] = pd.to_numeric(merged["fitzpatrick_scale"], errors="coerce")

tone_map = {
    1: "Type I – Very Fair",
    2: "Type II – Fair",
    3: "Type III – Light Brown",
    4: "Type IV – Moderate Brown",
    5: "Type V – Dark Brown",
    6: "Type VI – Deeply Pigmented",
}

def tone_group(t):
    if pd.isna(t) or t == -1:
        return "Unknown"
    return tone_map.get(int(t), "Unknown")

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)

# ordered categories for clean printing/plotting
order = list(tone_map.values()) + ["Unknown"]
merged["tone_group"] = pd.Categorical(merged["tone_group"], categories=order, ordered=True)

print("\nTone counts:")
print(merged["tone_group"].value_counts(dropna=False).reindex(order))

# ---- summary ----
tone_summary = (
    merged.groupby("tone_group", observed=True)[["bertscore_recall_kw_clef", "exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Fitzpatrick Tone (I–VI + Unknown) ===")
print(tone_summary)

# ---- save ----
out_path = f"{WORK_DIR}/bertscore_kw_clef_with_fitzpatrick_names.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)

Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Tone counts:
tone_group
Type I – Very Fair            588
Type II – Fair                996
Type III – Light Brown        634
Type IV – Moderate Brown      534
Type V – Dark Brown           322
Type VI – Deeply Pigmented    135
Unknown                       107
Name: count, dtype: int64

=== Average Caption Quality by Fitzpatrick Tone (I–VI + Unknown) ===
                   tone_group  bertscore_recall_kw_clef  \
0          Type I – Very Fair                  0.521085   
1              Type II – Fair                  0.518487   
2      Type III – Light Brown                  0.523633   
3    Type IV – Moderate Brown                  0.527707   
4         Type V – Dark Brown                  0.533982   
5  Type VI – Deeply Pigmented                  0.529924   
6                     Unknown                  0.528415   

   exact_keyword_coverage  
0                0.162356  
1            

In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/smolvlm_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# merge key
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# filter invalid
merged = merged[merged["fitzpatrick_scale"] > 0].copy()

def tone_group(t):
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)
print("\nTone counts:")
print(merged["tone_group"].value_counts())

tone_summary = (
    merged.groupby("tone_group")[["bertscore_recall_kw_clef","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone (CLEF-style) ===")
print(tone_summary)

out_path = f"{WORK_DIR}/bertscore_kw_clef_with_tone.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)


Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Tone counts:
tone_group
Light     1584
Medium    1168
Dark       457
Name: count, dtype: int64

=== Average Caption Quality by Skin Tone (CLEF-style) ===
  tone_group  bertscore_recall_kw_clef  exact_keyword_coverage
0       Dark                  0.532783                0.210757
1      Light                  0.519452                0.165007
2     Medium                  0.525495                0.197534

Saved merged dataset: /content/bertscore_kw_clef_with_tone.csv


In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/smolvlm_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# merge key
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# keep -1 (Unknown), just ensure numeric
merged["fitzpatrick_scale"] = pd.to_numeric(merged["fitzpatrick_scale"], errors="coerce")

tone_map = {
    1: "Type I – Very Fair",
    2: "Type II – Fair",
    3: "Type III – Light Brown",
    4: "Type IV – Moderate Brown",
    5: "Type V – Dark Brown",
    6: "Type VI – Deeply Pigmented",
}

def tone_group(t):
    if pd.isna(t) or t == -1:
        return "Unknown"
    return tone_map.get(int(t), "Unknown")

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)

# ordered categories (nice printing/plots)
order = list(tone_map.values()) + ["Unknown"]
merged["tone_group"] = pd.Categorical(merged["tone_group"], categories=order, ordered=True)

print("\nTone counts:")
print(merged["tone_group"].value_counts(dropna=False).reindex(order))

tone_summary = (
    merged.groupby("tone_group", observed=True)[["bertscore_recall_kw_clef", "exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Fitzpatrick Tone (I–VI + Unknown) ===")
print(tone_summary)

out_path = f"{WORK_DIR}/smolvlm_bertscore_kw_clef_with_6tone.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)

Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Tone counts:
tone_group
Type I – Very Fair            588
Type II – Fair                996
Type III – Light Brown        634
Type IV – Moderate Brown      534
Type V – Dark Brown           322
Type VI – Deeply Pigmented    135
Unknown                       107
Name: count, dtype: int64

=== Average Caption Quality by Fitzpatrick Tone (I–VI + Unknown) ===
                   tone_group  bertscore_recall_kw_clef  \
0          Type I – Very Fair                  0.521085   
1              Type II – Fair                  0.518487   
2      Type III – Light Brown                  0.523633   
3    Type IV – Moderate Brown                  0.527707   
4         Type V – Dark Brown                  0.533982   
5  Type VI – Deeply Pigmented                  0.529924   
6                     Unknown                  0.528415   

   exact_keyword_coverage  
0                0.162356  
1            

In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/smolvlm_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# ---- merge key ----
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# ---- ensure numeric ----
merged["fitzpatrick_scale"] = pd.to_numeric(
    merged["fitzpatrick_scale"], errors="coerce"
)

# ---- count Unknown ----
unknown_count = merged[
    merged["fitzpatrick_scale"].isna() | (merged["fitzpatrick_scale"] == -1)
].shape[0]

print(f"\nUnknown Fitzpatrick samples (excluded from fairness analysis): {unknown_count}")

# ---- keep only valid tones 1–6 ----
analysis_df = merged[merged["fitzpatrick_scale"].isin([1,2,3,4,5,6])].copy()

# ---- 6-tone mapping ----
tone_map = {
    1: "Type I – Very Fair",
    2: "Type II – Fair",
    3: "Type III – Light Brown",
    4: "Type IV – Moderate Brown",
    5: "Type V – Dark Brown",
    6: "Type VI – Deeply Pigmented",
}

analysis_df["tone_group"] = analysis_df["fitzpatrick_scale"].map(tone_map)

# ordered tones
order = list(tone_map.values())
analysis_df["tone_group"] = pd.Categorical(
    analysis_df["tone_group"],
    categories=order,
    ordered=True
)

print("\nTone counts (I–VI only):")
print(analysis_df["tone_group"].value_counts().reindex(order))

# ---- fairness summary ----
tone_summary = (
    analysis_df.groupby("tone_group")[["bertscore_recall_kw_clef", "exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Fitzpatrick Tone (I–VI) ===")
print(tone_summary)

# ---- save ----
out_path = f"{WORK_DIR}/smolvlm_bertscore_kw_clef_with_6tone_clean.csv"
analysis_df.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)

Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Unknown Fitzpatrick samples (excluded from fairness analysis): 107

Tone counts (I–VI only):
tone_group
Type I – Very Fair            588
Type II – Fair                996
Type III – Light Brown        634
Type IV – Moderate Brown      534
Type V – Dark Brown           322
Type VI – Deeply Pigmented    135
Name: count, dtype: int64

=== Average Caption Quality by Fitzpatrick Tone (I–VI) ===
                   tone_group  bertscore_recall_kw_clef  \
0          Type I – Very Fair                  0.521085   
1              Type II – Fair                  0.518487   
2      Type III – Light Brown                  0.523633   
3    Type IV – Moderate Brown                  0.527707   
4         Type V – Dark Brown                  0.533982   
5  Type VI – Deeply Pigmented                  0.529924   

   exact_keyword_coverage  
0                0.162356  
1                0.166572  
2       

/tmp/ipython-input-2383884331.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis_df.groupby("tone_group")[["bertscore_recall_kw_clef", "exact_keyword_coverage"]]


In [ ]:
import pandas as pd

bert_path = f"{WORK_DIR}/smolvlm_caption_vs_keywords_CLEF_BERTScore.csv"
fitz_path = f"{WORK_DIR}/fitzpatrick17k - fitzpatrick17k.csv"

bert = pd.read_csv(bert_path)
fitz = pd.read_csv(fitz_path)

print("Loaded files:")
print(f"- BERTScore file: {len(bert)} rows")
print(f"- Fitzpatrick file: {len(fitz)} rows")

# ---- merge key ----
bert["key"] = bert["key"].astype(str).str.lower()
fitz["key"] = fitz["md5hash"].astype(str).str.lower()

merged = bert.merge(fitz, on="key", how="inner")
print(f"Merged dataset size: {len(merged)}")

# ---- ensure numeric ----
merged["fitzpatrick_scale"] = pd.to_numeric(
    merged["fitzpatrick_scale"], errors="coerce"
)

# ---- count Unknown ----
unknown_count = merged[
    merged["fitzpatrick_scale"].isna() | (merged["fitzpatrick_scale"] == -1)
].shape[0]

print(f"\nUnknown Fitzpatrick samples: {unknown_count}")

# ---- tone grouping (Light/Medium/Dark only) ----
def tone_group(t):
    if pd.isna(t) or t == -1:
        return "Unknown"
    t = int(t)
    if t in [1, 2]:
        return "Light"
    elif t in [3, 4]:
        return "Medium"
    elif t in [5, 6]:
        return "Dark"
    return "Unknown"

merged["tone_group"] = merged["fitzpatrick_scale"].apply(tone_group)

print("\nTone counts (including Unknown):")
print(merged["tone_group"].value_counts())

# ---- exclude Unknown from fairness metrics ----
analysis_df = merged[merged["tone_group"] != "Unknown"]

tone_summary = (
    analysis_df.groupby("tone_group")[["bertscore_recall_kw_clef","exact_keyword_coverage"]]
    .mean()
    .reset_index()
)

print("\n=== Average Caption Quality by Skin Tone (Light/Medium/Dark) ===")
print(tone_summary)

# ---- save ----
out_path = f"{WORK_DIR}/smol_bertscore_kw_clef_with_tone_clean.csv"
merged.to_csv(out_path, index=False)
print("\nSaved merged dataset:", out_path)

Loaded files:
- BERTScore file: 3316 rows
- Fitzpatrick file: 16577 rows
Merged dataset size: 3316

Unknown Fitzpatrick samples: 107

Tone counts (including Unknown):
tone_group
Light      1584
Medium     1168
Dark        457
Unknown     107
Name: count, dtype: int64

=== Average Caption Quality by Skin Tone (Light/Medium/Dark) ===
  tone_group  bertscore_recall_kw_clef  exact_keyword_coverage
0       Dark                  0.532783                0.210757
1      Light                  0.519452                0.165007
2     Medium                  0.525495                0.197534

Saved merged dataset: /content/smol_bertscore_kw_clef_with_tone_clean.csv
